# Alpha 因子暴露

计算 33 个候选 Alpha，分别完成截面处理与风格中性化，保存 `alpha_factor_exposure.parquet`。运行前需先生成行情面板、指数行情和风格暴露。


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

## 公共计算函数

`wz` 完成按日缩尾和标准化；`rolling_corr` 要求完整配对窗口，将恒定序列及无效相关性置空，并限制浮点误差造成的越界。


In [ ]:
def wz(s):
    s = s.clip(lower=s.quantile(0.01), upper= s.quantile(0.99))
    s = (s - s.mean()) / s.std(ddof=0)
    return s

In [ ]:
def rolling_corr(x, y, window):
    x_window = x.rolling(window, min_periods=window)
    y_window = y.rolling(window, min_periods=window)
    corr = x_window.corr(y)
    constant = (x_window.max() == x_window.min()) | (y_window.max() == y_window.min())
    corr.loc[constant] = np.nan
    return corr.replace([np.inf, -np.inf], np.nan).clip(-1, 1)

## 文件路径与参数

自动向上定位项目根目录，使用项目相对路径读写文件；内核工作目录应位于本项目内。下方同时定义本步骤使用的参数。


In [ ]:
PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'environment.yml').is_file() and (path / 'src').is_dir()
)

PANEL = PROJECT_ROOT / "data/processed/panel.parquet"
INDEX_ = PROJECT_ROOT / "data/processed/index.parquet"
STYLE_FACTOR = PROJECT_ROOT / "src/factor_exposure/style_factor/style_factor_exposure.parquet"
ALPHA_FACTOR_OUTPUT = PROJECT_ROOT / "src/factor_exposure/alpha_factor/alpha_factor_exposure.parquet"


## 读取输入数据

读取面板、指数和八个风格暴露。面板沿用上游日期顺序，指数按交易日期与股票记录匹配。


In [ ]:
panel = pd.read_parquet(PANEL)
index = pd.read_parquet(INDEX_)
style_factor_exposure = pd.read_parquet(STYLE_FACTOR)

In [ ]:
panel.head()

In [ ]:
index.head()

## 初始化因子结果

每个 Alpha 依次选取输入、计算原始值、保留三列、执行截面 `wz` 并加入 `alpha_exp_lis`。下方按类型分组，公式说明同时列明停牌、填充及最低记录要求。


In [ ]:
alpha_exp = panel[['trade_date', 'ts_code']].copy()
alpha_exp_lis = []

## 因子计算

按经济含义与计算结构分组，各因子保留独立的说明及代码单元格。沿用已确认的停牌、缺失值和代理窗口口径。

滚动相关统一使用 `rolling_corr`：完整配对窗口、恒定序列置空、非有限结果置空，并将浮点误差导致的越界值限制在 [-1, 1]。

## 价格趋势与相对强弱

### Alpha15

**衡量内容（原始值）：** 衡量当天开盘价相对上一交易日参考收盘价的跳空幅度。原始值为正表示高开，为负表示低开，绝对值越大表示跳空幅度越大；若前一交易日停牌，则相对停牌前最后有效收盘价计算。

$$
\mathrm{Alpha15}_{i,t}=\frac{O_{i,t}}{\widetilde C_{i,t-1}}-1
$$

$O$ 为 `open_hfq`，$C$ 为 `close_hfq`，$\widetilde C$ 为按股票、按日期前向填充后的收盘价序列。

填充方法：在每只股票内部对收盘价执行 `ffill()`，再滞后 1 个交易日作为分母；当天开盘价保持原值。复牌日使用停牌前最后有效收盘价。首次有效收盘价之前不填充，当天开盘价缺失时结果为 `NaN`。

**最少记录数：** 至少需要 2 个交易日记录；当天开盘价有效，前一交易日或更早已有有效收盘价。记录数按市场交易日计，包含停牌日；上述条件指原始因子可计算，截面 `wz` 还需当日有效样本的标准差非零。

In [ ]:
alpha15 = panel[['trade_date', 'ts_code', 'open_hfq', 'close_hfq']].copy()

In [ ]:
alpha15['close_hfq_lag1'] = (
    alpha15.groupby('ts_code', sort=False)['close_hfq']
    .transform(lambda s: s.ffill().shift(1))
)

alpha15['alpha15'] = (
    alpha15['open_hfq'] / alpha15['close_hfq_lag1'] - 1
)

In [ ]:
alpha15 = alpha15[['trade_date', 'ts_code', 'alpha15']]

In [ ]:
alpha15.head()

In [ ]:
alpha15['alpha15'] = (
    alpha15.groupby('trade_date')['alpha15'].transform(wz)
)

alpha_exp_lis.append(alpha15)

### Alpha20

**衡量内容（原始值）：** 衡量最近 6 个交易日的收盘价累计涨跌幅（百分数）。原始值越高，表示这一期间价格表现越强；只反映起止价格变化，不反映期间上涨次数或路径。参考日停牌时使用截至该日最后有效收盘价，实际收益跨度可能超过 6 个交易日。

$$
\mathrm{Alpha20}_{i,t}=100\left(\frac{C_{i,t}}{\widetilde C_{i,t-6}}-1\right)
$$

$C$ 为 `close_hfq`，$\widetilde C$ 为按股票、按日期前向填充后的收盘价序列。

填充方法：在每只股票内部对收盘价执行 `ffill()`，再滞后 6 个交易日作为分母；当天收盘价保持原值。如果第 6 个交易日前的收盘价缺失，使用截至该日最后有效的收盘价，不跳过停牌日计数。首次有效收盘价之前不填充，当天收盘价缺失时结果为 `NaN`。

**最少记录数：** 至少需要 8 个交易日记录；当天收盘价有效，且第 6 个交易日前或更早已有有效收盘价。记录数按市场交易日计，包含停牌日；上述条件指原始因子可计算，截面 `wz` 还需当日有效样本的标准差非零。

In [ ]:
alpha20 = panel[['trade_date', 'ts_code', 'close_hfq']].copy()

In [ ]:
alpha20['close_hfq_lag6'] = (
    alpha20.groupby('ts_code', sort=False)['close_hfq']
    .transform(lambda s: s.ffill().shift(6))
)

alpha20['alpha20'] = (
    alpha20['close_hfq'] / alpha20['close_hfq_lag6'] - 1
) * 100

In [ ]:
alpha20 = alpha20[['trade_date', 'ts_code', 'alpha20']]

In [ ]:
alpha20.head()

In [ ]:
alpha20['alpha20'] = (
    alpha20.groupby('trade_date')['alpha20'].transform(wz)
)

alpha_exp_lis.append(alpha20)

### Alpha88

**衡量内容（原始值）：** 衡量最近 20 个交易日的收盘价累计涨跌幅（百分数）。原始值越高，表示这一期间价格表现越强，可用于描述比 Alpha20 更长的价格趋势。参考日停牌时使用截至该日最后有效收盘价，实际收益跨度可能超过 20 个交易日。

$$
\mathrm{Alpha88}_{i,t}=100\left(\frac{C_{i,t}}{\widetilde C_{i,t-20}}-1\right)
$$

$C$ 为 `close_hfq`，$\widetilde C$ 为按股票、按日期前向填充后的收盘价序列。

填充方法：在每只股票内部对收盘价执行 `ffill()`，再滞后 20 个交易日作为分母；当天收盘价保持原值。如果第 20 个交易日前的收盘价缺失，使用截至该日最后有效的收盘价，不跳过停牌日计数。首次有效收盘价之前不填充，当天收盘价缺失时结果为 `NaN`。

**最少记录数：** 至少需要 21 个交易日记录；当天收盘价有效，且第 20 个交易日前或更早已有有效收盘价。记录数按市场交易日计，包含停牌日；上述条件指原始因子可计算，截面 `wz` 还需当日有效样本的标准差非零。

In [ ]:
alpha88 = panel[['trade_date', 'ts_code', 'close_hfq']].copy()

In [ ]:
alpha88['close_hfq_lag20'] = (
    alpha88.groupby('ts_code', sort=False)['close_hfq']
    .transform(lambda s: s.ffill().shift(20))
)

alpha88['alpha88'] = (
    alpha88['close_hfq'] / alpha88['close_hfq_lag20'] - 1
) * 100

In [ ]:
alpha88 = alpha88[['trade_date', 'ts_code', 'alpha88']]

In [ ]:
alpha88.head()

In [ ]:
alpha88['alpha88'] = (
    alpha88.groupby('trade_date')['alpha88'].transform(wz)
)

alpha_exp_lis.append(alpha88)

### Alpha31

**衡量内容（原始值）：** 衡量当天收盘价相对最近 12 个交易日均价的偏离程度（百分数）。原始值为正表示高于均价，为负表示低于均价，绝对值越大表示偏离越明显。均价中的停牌日使用最后有效收盘价；该指标本身不保证偏离后一定反转。

$$
\overline C^{(12)}_{i,t}=\frac{1}{12}\sum_{k=0}^{11}\widetilde C_{i,t-k}
$$

$$
\mathrm{Alpha31}_{i,t}=100\frac{C_{i,t}-\overline C^{(12)}_{i,t}}{\overline C^{(12)}_{i,t}}
$$

$C$ 为 `close_hfq`，$\widetilde C$ 为按股票、按日期前向填充后的收盘价序列。

填充方法：在每只股票内部先对收盘价执行 `ffill()`，仅用于计算包含当天的 12 个交易日滚动均值，设置 `min_periods=12`；公式中的当天收盘价保持原值。停牌日在均值中沿用最后有效收盘价，但当天因子仍为 `NaN`。首次有效收盘价之前不填充，窗口不足 12 个有效价格时结果为 `NaN`。

**最少记录数：** 至少需要 12 个交易日记录；窗口内前向填充后须有 12 个有效收盘价，且当天收盘价有效。记录数按市场交易日计，包含停牌日；上述条件指原始因子可计算，截面 `wz` 还需当日有效样本的标准差非零。

In [ ]:
alpha31 = panel[['trade_date', 'ts_code', 'close_hfq']].copy()

In [ ]:
alpha31['close_mean12'] = (
    alpha31.groupby('ts_code', sort=False)['close_hfq']
    .transform(
        lambda s: s.ffill().rolling(12, min_periods=12).mean()
    )
)

alpha31['alpha31'] = (
    alpha31['close_hfq'] / alpha31['close_mean12'] - 1
) * 100

In [ ]:
alpha31 = alpha31[['trade_date', 'ts_code', 'alpha31']]

In [ ]:
alpha31.head()

In [ ]:
alpha31['alpha31'] = (
    alpha31.groupby('trade_date')['alpha31'].transform(wz)
)

alpha_exp_lis.append(alpha31)

### Alpha53

**衡量内容（原始值）：** 衡量最近 12 个市场交易日中收盘价上涨的频率，原始值为上涨天数占比乘以 100。数值越高表示上涨日越多，不代表累计涨幅越大；平盘与停牌日均不计为上涨，停牌日仍占用窗口，可能压低复牌后的上涨频率。

$$
\widetilde C_{i,t}=\operatorname{FFill}_i(C_{i,t}),\qquad
U_{i,t}=\mathbf{1}\left(\widetilde C_{i,t}>\widetilde C_{i,t-1}\right)
$$

$$
\mathrm{Alpha53}^{\mathrm{raw}}_{i,t}
=\frac{100}{12}\sum_{k=0}^{11}U_{i,t-k}
$$

$C$ 为后复权收盘价 `close_hfq`。因子表示最近 12 个市场交易日中上涨天数的百分比，只衡量上涨频率，不衡量涨幅。

填充方法：按股票前向填充收盘价后，与前一个交易日比较；停牌日价格不变，记为未上涨，平盘同样记为未上涨。复牌日与停牌前最后有效收盘价比较，不跳过停牌日计数。当天或前一日参考价格无效时，上涨标记为 `NaN`，不将缺失比较记为未上涨。采用完整 12 日窗口（`min_periods=12`），当天停牌时按 `suspend` 将因子置为 `NaN`，加入列表前按 `trade_date` 执行截面 `wz`。

**最少记录数：** 至少需要 13 个交易日记录，形成 12 次有效价格比较；最早从首个有效收盘价所在日之后的第 12 个交易日开始输出，且当天须非停牌。上市前或数据起点前的历史不会自动补齐。截面 `wz` 还需当日有效样本的标准差非零。

In [ ]:
alpha53 = panel[['trade_date', 'ts_code', 'close_hfq', 'suspend']].copy()

In [ ]:
alpha53['close_filled'] = (
    alpha53.groupby('ts_code', sort=False)['close_hfq'].ffill()
)

alpha53['close_lag1'] = (
    alpha53.groupby('ts_code', sort=False)['close_filled'].shift(1)
)

alpha53['up'] = (
    alpha53['close_filled'] > alpha53['close_lag1']
).astype(float)

alpha53.loc[
    alpha53['close_filled'].isna() | alpha53['close_lag1'].isna(),
    'up'
] = np.nan

alpha53['alpha53'] = (
    alpha53.groupby('ts_code', sort=False)['up']
    .transform(lambda s: s.rolling(12, min_periods=12).sum())
) * 100 / 12

alpha53.loc[alpha53['suspend'] == True, 'alpha53'] = np.nan

In [ ]:
alpha53 = alpha53[['trade_date', 'ts_code', 'alpha53']]

In [ ]:
alpha53.head()

In [ ]:
alpha53['alpha53'] = (
    alpha53.groupby('trade_date')['alpha53'].transform(wz)
)

alpha_exp_lis.append(alpha53)

### Alpha63

**衡量内容（原始值）：** 衡量递归平滑后的上涨价格变动占全部绝对价格变动的比例，属于 RSI 类指标。原始值通常介于 0 至 100：接近 100 表示上涨幅度占优，接近 0 表示下跌幅度占优，50 表示平滑后的上涨、下跌幅度大致相等。它不保证后续反转。与使用固定窗口累计涨跌幅的 Alpha112 不同，本因子采用递归平滑，历史影响随有效更新逐渐衰减；输入为绝对价格变化，而非百分比收益率。

$$
\widetilde C_{i,t}=\operatorname{FFill}_i(C_{i,t}),\qquad
\Delta C_{i,t}=C_{i,t}-\widetilde C_{i,t-1}
$$

$$
U_{i,t}=\max(\Delta C_{i,t},0),\qquad A_{i,t}=|\Delta C_{i,t}|
$$

$$
\mathrm{Alpha63}^{\mathrm{raw}}_{i,t}
=100\frac{\operatorname{SMA}(U,6,1)_{i,t}}
{\operatorname{SMA}(A,6,1)_{i,t}}
$$

$C$ 为后复权收盘价 `close_hfq`。仅将历史参考收盘价按股票前向填充后滞后 1 日，当天收盘价保留原值。复牌日与停牌前最后有效收盘价比较；首次没有有效参考价格时价差保留 `NaN`。停牌日按 `suspend` 将两个平滑输入均置空，不将停牌价差 0 输入递推。

**初始化和递推：** 两个平滑序列分别以首个有效输入初始化，此后每个有效日使用 $S_{i,t}=X_{i,t}/6+5S_{i,t'}/6$ 更新，$t'$ 为最后一次有效更新日。代码采用 `ewm(alpha=1/6, adjust=False, ignore_na=True)`。输入缺失时保留内部状态、暂停更新，但当天因子输出为 `NaN`；复牌日输入有效即可恢复更新，不额外等待 6 日。若首个价差为 0，两个状态均初始化为 0，分母未大于 0 前比值无定义。

分母为 0 时结果设为 `NaN`。两个递归状态使用未标准化的输入和状态；最后才按 `trade_date` 执行截面 `wz`，再加入列表。

**最少记录数：** 最早需要 2 个交易日价格记录，且当天非停牌、当前及参考收盘价有效，平滑分母大于 0。首个非零价差初始化后结果可能为 0 或 100。参数 6 是递归平滑参数，不是完整滚动窗口长度；初值影响随有效更新次数按 $(5/6)^n$ 衰减。截面 `wz` 还需当日有效样本的标准差非零。

In [ ]:
alpha63 = panel[['trade_date', 'ts_code', 'close_hfq', 'suspend']].copy()

In [ ]:
alpha63['close_lag1'] = (
    alpha63.groupby('ts_code', sort=False)['close_hfq']
    .transform(lambda s: s.ffill().shift(1))
)

alpha63['price_change'] = alpha63['close_hfq'] - alpha63['close_lag1']
alpha63.loc[alpha63['suspend'] == True, 'price_change'] = np.nan

alpha63['up_move'] = alpha63['price_change'].clip(lower=0)
alpha63['abs_move'] = alpha63['price_change'].abs()

alpha63['up_sma6'] = (
    alpha63.groupby('ts_code', sort=False)['up_move']
    .transform(lambda s: s.ewm(alpha=1/6, adjust=False, ignore_na=True).mean())
)

alpha63['abs_sma6'] = (
    alpha63.groupby('ts_code', sort=False)['abs_move']
    .transform(lambda s: s.ewm(alpha=1/6, adjust=False, ignore_na=True).mean())
)

alpha63['alpha63'] = (
    alpha63['up_sma6'] / alpha63['abs_sma6'].replace(0, np.nan)
) * 100

alpha63.loc[alpha63['price_change'].isna(), 'alpha63'] = np.nan

In [ ]:
alpha63 = alpha63[['trade_date', 'ts_code', 'alpha63']]

In [ ]:
alpha63.head()

In [ ]:
alpha63['alpha63'] = (
    alpha63.groupby('trade_date')['alpha63'].transform(wz)
)

alpha_exp_lis.append(alpha63)

### Alpha112

**衡量内容（原始值）：** 衡量最近 12 个市场交易日中，累计上涨价格幅度与累计下跌价格幅度的差额，占两者总和的比例。原始值介于 -100 与 100 之间：正值表示累计上涨幅度占优，负值表示累计下跌幅度占优，0 表示两者相等；仅有上涨且累计涨幅大于 0 时为 100，仅有下跌且累计跌幅大于 0 时为 -100。它使用收盘价的绝对价差而非百分比收益率，不计成交量，也不等于累计收益率。与 Alpha53 的上涨天数占比相比，本因子考虑涨跌的幅度。

$$
\widetilde C_{i,t}=\operatorname{FFill}_i(C_{i,t}),\qquad
\Delta C_{i,t}=\widetilde C_{i,t}-\widetilde C_{i,t-1}
$$

$$
U_{i,t}^{(12)}=\sum_{k=0}^{11}\max(\Delta C_{i,t-k},0),\qquad
D_{i,t}^{(12)}=\sum_{k=0}^{11}\max(-\Delta C_{i,t-k},0)
$$

$$
\mathrm{Alpha112}^{\mathrm{raw}}_{i,t}
=100\frac{U_{i,t}^{(12)}-D_{i,t}^{(12)}}{U_{i,t}^{(12)}+D_{i,t}^{(12)}}
$$

$C$ 为后复权收盘价 `close_hfq`。按股票前向填充收盘价，再计算相邻市场交易日的价差；停牌日价格不变，涨跌贡献均为 0，复牌日相对停牌前最后有效收盘价计算。不填充首次有效收盘价之前的数据，首次缺少参考价格的价差保留 `NaN`。

采用包含当天的完整 12 日窗口（`min_periods=12`），不跳过停牌日。累计上涨与下跌幅度之和为 0 时，因子无定义，设为 `NaN`。计算后按 `suspend == True` 将当天因子设为 `NaN`，加入列表前按 `trade_date` 执行截面 `wz`。

**最少记录数：** 至少需要 13 个交易日记录，形成 12 次有效价格比较；窗口内价差须有效，累计绝对价差须大于 0，且当天非停牌。截面 `wz` 还需当日有效样本的标准差非零。

In [ ]:
alpha112 = panel[['trade_date', 'ts_code', 'close_hfq', 'suspend']].copy()

In [ ]:
alpha112['close_filled'] = (
    alpha112.groupby('ts_code', sort=False)['close_hfq'].ffill()
)

alpha112['price_change'] = (
    alpha112.groupby('ts_code', sort=False)['close_filled'].diff()
)

alpha112['up_move'] = alpha112['price_change'].clip(lower=0)
alpha112['down_move'] = (-alpha112['price_change']).clip(lower=0)

alpha112['up_sum12'] = (
    alpha112.groupby('ts_code', sort=False)['up_move']
    .transform(lambda s: s.rolling(12, min_periods=12).sum())
)

alpha112['down_sum12'] = (
    alpha112.groupby('ts_code', sort=False)['down_move']
    .transform(lambda s: s.rolling(12, min_periods=12).sum())
)

alpha112['alpha112'] = (
    (alpha112['up_sum12'] - alpha112['down_sum12'])
    / (alpha112['up_sum12'] + alpha112['down_sum12']).replace(0, np.nan)
) * 100

alpha112.loc[alpha112['suspend'] == True, 'alpha112'] = np.nan

In [ ]:
alpha112 = alpha112[['trade_date', 'ts_code', 'alpha112']]

In [ ]:
alpha112.head()

In [ ]:
alpha112['alpha112'] = (
    alpha112.groupby('trade_date')['alpha112'].transform(wz)
)

alpha_exp_lis.append(alpha112)

### Alpha177

**衡量内容（原始值）：** 衡量最近 20 个市场交易日的最高价距离今天有多久，反映近期高点的新鲜程度。高点越近，原始值越高；当天触及窗口最高价时为 100，高点在 19 个交易日前时为 5。它只看高点出现时间，不衡量当前价格距离高点有多远，也不直接衡量涨跌幅；当天冲高回落仍可能得到高值。

$$
D_{i,t}=\operatorname{HIGHDAY}(H,20)_{i,t},\qquad
\mathrm{Alpha177}^{\mathrm{raw}}_{i,t}=100\frac{20-D_{i,t}}{20}
$$

$H$ 为后复权最高价 `high_hfq`。$D$ 为包含当天的 20 日窗口内，最高价出现日距今天的交易日数，当天记为 0，最早一天记为 19。重复最高价取最近一次出现的位置；代码将窗口倒序后用 `np.argmax`，使其返回距今天的天数。

最高价不填充。按股票使用完整 20 日窗口（`min_periods=20`），含缺失最高价的窗口结果为空，不跳过停牌日凑足记录。当天停牌按 `suspend == True` 将因子置为 `NaN`；加入列表前按 `trade_date` 执行截面 `wz`。原始值范围为 5 至 100，标准化后的值不再受该范围限制。

**最少记录数：** 至少需要 20 个交易日记录，且窗口内最高价全部有效、当天非停牌。由于不填充最高价，最早在连续有效的复牌第 20 个交易日恢复计算。截面 `wz` 还需当日有效样本的标准差非零。

In [ ]:
alpha177 = panel[['trade_date', 'ts_code', 'high_hfq', 'suspend']].copy()

In [ ]:
alpha177['highday20'] = (
    alpha177.groupby('ts_code', sort=False)['high_hfq']
    .transform(
        lambda s: s.rolling(20, min_periods=20).apply(
            lambda values: np.argmax(values[::-1]), raw=True
        )
    )
)

alpha177['alpha177'] = (20 - alpha177['highday20']) * 100 / 20

alpha177.loc[alpha177['suspend'] == True, 'alpha177'] = np.nan

In [ ]:
alpha177 = alpha177[['trade_date', 'ts_code', 'alpha177']]

In [ ]:
alpha177.head()

In [ ]:
alpha177['alpha177'] = (
    alpha177.groupby('trade_date')['alpha177'].transform(wz)
)

alpha_exp_lis.append(alpha177)

## 日内形态与价格波动

### Alpha2

**衡量内容（原始值）：** 衡量收盘价在当日高低价区间中的相对位置较前一天变化的负值。原始值为正表示收盘相对位置转弱，为负表示转强；有效价格下原始值介于 -2 与 2 之间。它描述相对收盘位置的变化，不直接衡量收盘收益率，反向打分也不保证后续发生价格反转。

$$
P_{i,t}=\frac{(C_{i,t}-L_{i,t})-(H_{i,t}-C_{i,t})}{H_{i,t}-L_{i,t}},\qquad
\mathrm{Alpha2}^{\mathrm{raw}}_{i,t}=-(P_{i,t}-P_{i,t-1})
$$

$C,H,L$ 分别为 `close_hfq`、`high_hfq`、`low_hfq`。因子衡量收盘价在当日高低价区间中相对位置变化的负值。

不填充价格或位置指标。当天高低价相等时分母无定义，位置指标设为 `NaN`。按股票对位置指标滞后 1 个交易日，当天或前一日指标缺失时因子为 `NaN`，因此停牌日及紧接停牌的复牌日均为空。加入列表前按 `trade_date` 执行截面 `wz`。

**最少记录数：** 至少需要 2 个交易日记录；连续两天高、低、收盘价均有效，且两天最高价均不等于最低价。记录数按市场交易日计，包含停牌日；上述条件指原始因子可计算，截面 `wz` 还需当日有效样本的标准差非零。

In [ ]:
alpha2 = panel[['trade_date', 'ts_code', 'close_hfq', 'high_hfq', 'low_hfq']].copy()

In [ ]:
alpha2['close_position'] = (
    (2 * alpha2['close_hfq'] - alpha2['high_hfq'] - alpha2['low_hfq'])
    / (alpha2['high_hfq'] - alpha2['low_hfq']).replace(0, np.nan)
)

alpha2['alpha2'] = -(
    alpha2['close_position']
    - alpha2.groupby('ts_code', sort=False)['close_position'].shift(1)
)

In [ ]:
alpha2 = alpha2[['trade_date', 'ts_code', 'alpha2']]

In [ ]:
alpha2.head()

In [ ]:
alpha2['alpha2'] = (
    alpha2.groupby('trade_date')['alpha2'].transform(wz)
)

alpha_exp_lis.append(alpha2)

### Alpha47

**衡量内容（原始值）：** 衡量收盘价距离最近 6 日最高价的距离占该窗口高低价区间的比例，再进行递归平滑。平滑前输入为 0 表示收在窗口最高价，为 100 表示收在窗口最低价。原始因子越高，平滑后的收盘位置越靠近近期区间低端；越低则越靠近高端。它描述相对收盘位置，不直接表示收益率，也不保证后续发生反转。

$$
H_{i,t}^{(6)}=\max_{0\le k\le5}H_{i,t-k},\qquad
L_{i,t}^{(6)}=\min_{0\le k\le5}L_{i,t-k}
$$

$$
X_{i,t}=100\frac{H_{i,t}^{(6)}-C_{i,t}}{H_{i,t}^{(6)}-L_{i,t}^{(6)}},\qquad
\mathrm{Alpha47}^{\mathrm{raw}}_{i,t}=\operatorname{SMA}(X,9,1)_{i,t}
$$

$C,H,L$ 分别为 `close_hfq`、`high_hfq`、`low_hfq`，均不填充。内层最高价、最低价使用完整 6 日窗口（`min_periods=6`）；高低区间为 0 或价格缺失时，输入 $X$ 为 `NaN`。停牌日的输入也按 `suspend` 置空。

**初始化和递推：** 每只股票以首个有效 $X$ 初始化内部状态 $S=X$。此后有效输入按下式更新，其中 $t'$ 为此前最后一次有效更新的日期：

$$
S_{i,t}=\frac{1}{9}X_{i,t}+\frac{8}{9}S_{i,t'}
$$

输入缺失时保留内部状态、不更新，当天输出为 `NaN`。代码使用 `ewm(alpha=1/9, adjust=False, ignore_na=True)` 维护状态，再按输入缺失标记屏蔽当日输出。复牌当天算第 1 日，最早第 6 日恢复有效输入，并从停牌前最后有效状态继续递推；若尚未初始化则以当天输入初始化。长时间停牌也不对保留状态额外衰减。

**最少记录数：** 最早需要 6 个连续有效的交易日高低价记录，且当天收盘价有效、非停牌、6 日区间非零，即可初始化。递归平滑没有严格有限的回看长度，初值影响随有效更新次数按 $(8/9)^n$ 衰减。加入列表前按 `trade_date` 执行截面 `wz`，递推使用未标准化的内部状态；截面 `wz` 还需当日有效样本的标准差非零。

In [ ]:
alpha47 = panel[['trade_date', 'ts_code', 'close_hfq', 'high_hfq', 'low_hfq', 'suspend']].copy()

In [ ]:
alpha47['high_max6'] = (
    alpha47.groupby('ts_code', sort=False)['high_hfq']
    .transform(lambda s: s.rolling(6, min_periods=6).max())
)

alpha47['low_min6'] = (
    alpha47.groupby('ts_code', sort=False)['low_hfq']
    .transform(lambda s: s.rolling(6, min_periods=6).min())
)

alpha47['sma_input'] = (
    (alpha47['high_max6'] - alpha47['close_hfq'])
    / (alpha47['high_max6'] - alpha47['low_min6']).replace(0, np.nan)
) * 100
alpha47.loc[alpha47['suspend'] == True, 'sma_input'] = np.nan

alpha47['alpha47'] = (
    alpha47.groupby('ts_code', sort=False)['sma_input']
    .transform(lambda s: s.ewm(alpha=1/9, adjust=False, ignore_na=True).mean())
)

alpha47.loc[alpha47['sma_input'].isna(), 'alpha47'] = np.nan

In [ ]:
alpha47 = alpha47[['trade_date', 'ts_code', 'alpha47']]

In [ ]:
alpha47.head()

In [ ]:
alpha47['alpha47'] = (
    alpha47.groupby('trade_date')['alpha47'].transform(wz)
)

alpha_exp_lis.append(alpha47)

### Alpha110

**衡量内容（原始值）：** 衡量最近 20 个市场交易日中，相对前一日参考收盘价的累计向上延伸幅度与累计向下延伸幅度之比。原始值为 100 表示两者相等，为 200 表示累计向上幅度为向下幅度的两倍。值越高，窗口内向上延伸相对越突出。该指标使用最高价与最低价，描述价格曾达到的范围，不要求当天最终收涨，不等同于累计收益率，也不使用成交量。

$$
\widetilde C_{i,t}=\operatorname{FFill}_i(C_{i,t})
$$

$$
U_{i,t}=\max(0,H_{i,t}-\widetilde C_{i,t-1}),\qquad
D_{i,t}=\max(0,\widetilde C_{i,t-1}-L_{i,t})
$$

$$
\mathrm{Alpha110}^{\mathrm{raw}}_{i,t}
=100\frac{\sum_{k=0}^{19}U_{i,t-k}}{\sum_{k=0}^{19}D_{i,t-k}}
$$

$C,H,L$ 分别为 `close_hfq`、`high_hfq`、`low_hfq`。按股票前向填充收盘价后滞后 1 个交易日，仅用于生成参考收盘价；当天最高价、最低价不填充。复牌日参考停牌前最后有效收盘价。非停牌日缺失参考价格时，向上、向下幅度保留 `NaN`，不替换为 0。

滚动统计时按 `suspend == True` 将停牌日的 $U,D$ 均设为 0，停牌日仍占用窗口。使用包含当天的完整 20 日窗口（`min_periods=20`），累计向下幅度为 0 时因子设为 `NaN`。计算后将当天停牌因子设为 `NaN`，加入列表前按 `trade_date` 执行截面 `wz`。

**最少记录数：** 通常至少需要 21 个交易日价格记录，形成 20 次有效的前收盘价比较；窗口内非停牌日须有有效最高价、最低价及参考收盘价，停牌日按零贡献计入窗口，累计向下幅度须大于 0，且当天非停牌。截面 `wz` 还需当日有效样本的标准差非零。

In [ ]:
alpha110 = panel[['trade_date', 'ts_code', 'close_hfq', 'high_hfq', 'low_hfq', 'suspend']].copy()

In [ ]:
alpha110['close_lag1'] = (
    alpha110.groupby('ts_code', sort=False)['close_hfq']
    .transform(lambda s: s.ffill().shift(1))
)

alpha110['up_range'] = (
    alpha110['high_hfq'] - alpha110['close_lag1']
).clip(lower=0)

alpha110['down_range'] = (
    alpha110['close_lag1'] - alpha110['low_hfq']
).clip(lower=0)

alpha110.loc[
    alpha110['suspend'] == True, ['up_range', 'down_range']
] = 0.0

alpha110['up_sum20'] = (
    alpha110.groupby('ts_code', sort=False)['up_range']
    .transform(lambda s: s.rolling(20, min_periods=20).sum())
)

alpha110['down_sum20'] = (
    alpha110.groupby('ts_code', sort=False)['down_range']
    .transform(lambda s: s.rolling(20, min_periods=20).sum())
)

alpha110['alpha110'] = (
    alpha110['up_sum20'] / alpha110['down_sum20'].replace(0, np.nan)
) * 100

alpha110.loc[alpha110['suspend'] == True, 'alpha110'] = np.nan

In [ ]:
alpha110 = alpha110[['trade_date', 'ts_code', 'alpha110']]

In [ ]:
alpha110.head()

In [ ]:
alpha110['alpha110'] = (
    alpha110.groupby('trade_date')['alpha110'].transform(wz)
)

alpha_exp_lis.append(alpha110)

### Alpha118

**衡量内容（原始值）：** 衡量最近 20 个市场交易日中，相对当天开盘价的累计向上延伸幅度与累计向下延伸幅度之比。原始值为 100 表示两者相等，为 200 表示累计向上幅度是向下幅度的两倍，值越高表示日内向上延伸相对更突出。与 Alpha110 参考前一日收盘价不同，本因子以当天开盘价为参考，不直接反映隔夜跳空。它使用最高价、最低价与开盘价，不使用收盘价或成交量，不等同于收盘收益率。

$$
U_{i,t}=H_{i,t}-O_{i,t},\qquad D_{i,t}=O_{i,t}-L_{i,t}
$$

$$
\mathrm{Alpha118}^{\mathrm{raw}}_{i,t}
=100\frac{\sum_{k=0}^{19}U_{i,t-k}}{\sum_{k=0}^{19}D_{i,t-k}}
$$

$O,H,L$ 分别为 `open_hfq`、`high_hfq`、`low_hfq`。价格不做填充。滚动统计时按 `suspend == True` 将停牌日的向上、向下幅度均设为 0，停牌日仍占用市场交易日窗口；非停牌日的缺失价格保留为缺失。

采用包含当天的完整 20 日窗口（`min_periods=20`），累计向下幅度为 0 时因子设为 `NaN`。计算后将当天停牌因子设为 `NaN`，加入列表前按 `trade_date` 执行截面 `wz`。

**最少记录数：** 至少需要 20 个交易日记录；窗口内非停牌日开、高、低价须有效，停牌日按零贡献计入窗口，累计向下幅度须大于 0，且当天非停牌。截面 `wz` 还需当日有效样本的标准差非零。

In [ ]:
alpha118 = panel[['trade_date', 'ts_code', 'open_hfq', 'high_hfq', 'low_hfq', 'suspend']].copy()

In [ ]:
alpha118['up_range'] = alpha118['high_hfq'] - alpha118['open_hfq']
alpha118['down_range'] = alpha118['open_hfq'] - alpha118['low_hfq']

alpha118.loc[
    alpha118['suspend'] == True, ['up_range', 'down_range']
] = 0.0

alpha118['up_sum20'] = (
    alpha118.groupby('ts_code', sort=False)['up_range']
    .transform(lambda s: s.rolling(20, min_periods=20).sum())
)

alpha118['down_sum20'] = (
    alpha118.groupby('ts_code', sort=False)['down_range']
    .transform(lambda s: s.rolling(20, min_periods=20).sum())
)

alpha118['alpha118'] = (
    alpha118['up_sum20'] / alpha118['down_sum20'].replace(0, np.nan)
) * 100

alpha118.loc[alpha118['suspend'] == True, 'alpha118'] = np.nan

In [ ]:
alpha118 = alpha118[['trade_date', 'ts_code', 'alpha118']]

In [ ]:
alpha118.head()

In [ ]:
alpha118['alpha118'] = (
    alpha118.groupby('trade_date')['alpha118'].transform(wz)
)

alpha_exp_lis.append(alpha118)

### Alpha161

**衡量内容（原始值）：** 衡量最近 12 个市场交易日的平均真实波幅。真实波幅同时考虑当天最高价与最低价的差距，以及当天高低价相对前一日参考收盘价的偏离，因此能够包含隔夜跳空。原始值越高表示绝对价格波动范围越大，不区分上涨或下跌。该指标未除以价格，单位与后复权价格相同，受股票价格水平及复权尺度影响，不等同于收益率波动率。

$$
\widetilde C_{i,t}=\operatorname{FFill}_i(C_{i,t})
$$

$$
\mathrm{TR}_{i,t}=\max\left(
H_{i,t}-L_{i,t},\;
|H_{i,t}-\widetilde C_{i,t-1}|,\;
|L_{i,t}-\widetilde C_{i,t-1}|
\right)
$$

$$
\widetilde{\mathrm{TR}}_{i,t}=\begin{cases}
0,&\mathrm{suspend}_{i,t}=\mathrm{True},\\
\mathrm{TR}_{i,t},&\text{其他情况},
\end{cases}
\qquad
\mathrm{Alpha161}^{\mathrm{raw}}_{i,t}
=\frac{1}{12}\sum_{k=0}^{11}\widetilde{\mathrm{TR}}_{i,t-k}
$$

$C,H,L$ 分别为 `close_hfq`、`high_hfq`、`low_hfq`。按股票前向填充收盘价后滞后 1 个交易日，仅用于生成参考收盘价；当天最高价、最低价不填充。复牌日参考停牌前最后有效收盘价。非停牌日若缺少参考收盘价或高低价，则真实波幅保留 `NaN`，不跳过缺失项求最大值。

滚动统计时按 `suspend == True` 将停牌日真实波幅设为 0，仍占用交易日窗口并计入均值分母，可能压低复牌后的均值。采用包含当天的完整 12 日简单平均（`min_periods=12`），不使用递归平滑。计算后将当天停牌因子设为 `NaN`，加入列表前按 `trade_date` 执行截面 `wz`。

**最少记录数：** 通常至少需要 13 个交易日价格记录，形成 12 个有效真实波幅；窗口内非停牌日须有有效高低价及参考收盘价，停牌日按零贡献计入窗口，且当天非停牌。截面 `wz` 还需当日有效样本的标准差非零。

In [ ]:
alpha161 = panel[['trade_date', 'ts_code', 'close_hfq', 'high_hfq', 'low_hfq', 'suspend']].copy()

In [ ]:
alpha161['close_lag1'] = (
    alpha161.groupby('ts_code', sort=False)['close_hfq']
    .transform(lambda s: s.ffill().shift(1))
)

alpha161['true_range'] = np.maximum(
    alpha161['high_hfq'] - alpha161['low_hfq'],
    np.maximum(
        (alpha161['high_hfq'] - alpha161['close_lag1']).abs(),
        (alpha161['low_hfq'] - alpha161['close_lag1']).abs()
    )
)

alpha161.loc[alpha161['suspend'] == True, 'true_range'] = 0.0

alpha161['alpha161'] = (
    alpha161.groupby('ts_code', sort=False)['true_range']
    .transform(lambda s: s.rolling(12, min_periods=12).mean())
)

alpha161.loc[alpha161['suspend'] == True, 'alpha161'] = np.nan

In [ ]:
alpha161 = alpha161[['trade_date', 'ts_code', 'alpha161']]

In [ ]:
alpha161.head()

In [ ]:
alpha161['alpha161'] = (
    alpha161.groupby('trade_date')['alpha161'].transform(wz)
)

alpha_exp_lis.append(alpha161)

### Alpha189

**衡量内容（原始值）：** 衡量最近 6 个市场交易日中，收盘价偏离各自当日 6 日均价的绝对距离的平均水平。原始值越高，表示价格与自身短期均线的距离通常越大，可能来自持续趋势或剧烈波动；值越低表示价格更贴近均线。该指标没有涨跌方向，不直接表示超买或超卖。它未按价格归一化，单位与后复权价格相同，受价格水平及复权尺度影响。与 Alpha31 的当天百分比偏离不同，本因子对多天绝对偏离距离取平均。

$$
\widetilde C_{i,t}=\operatorname{FFill}_i(C_{i,t}),\qquad
M_{i,t}^{(6)}=\frac{1}{6}\sum_{j=0}^{5}\widetilde C_{i,t-j}
$$

$$
\mathrm{Alpha189}^{\mathrm{raw}}_{i,t}
=\frac{1}{6}\sum_{k=0}^{5}\left|\widetilde C_{i,t-k}-M_{i,t-k}^{(6)}\right|
$$

$C$ 为后复权收盘价 `close_hfq`。按股票前向填充收盘价，用于内层 6 日均价和每天的绝对偏离值；不填充首次有效价格之前的数据。每个偏离值使用该日对应的滚动均价，不用今天的均价统一衡量过去各日。

内层均价与外层偏离值平均均使用包含当天的完整 6 日窗口（`min_periods=6`）。停牌日沿用最后有效价格，其偏离值可能非零，按实际计算值参与外层滚动平均，不强制设为 0。计算后按 `suspend == True` 将当天因子设为 `NaN`，加入列表前按 `trade_date` 执行截面 `wz`。

**最少记录数：** 至少需要 11 个交易日记录；从首个有效收盘价开始，第 6 日得到首个有效偏离值，第 11 日得到 6 个有效偏离值的平均。两层窗口均须完整，且当天非停牌。截面 `wz` 还需当日有效样本的标准差非零。

In [ ]:
alpha189 = panel[['trade_date', 'ts_code', 'close_hfq', 'suspend']].copy()

In [ ]:
alpha189['close_filled'] = (
    alpha189.groupby('ts_code', sort=False)['close_hfq'].ffill()
)

alpha189['close_mean6'] = (
    alpha189.groupby('ts_code', sort=False)['close_filled']
    .transform(lambda s: s.rolling(6, min_periods=6).mean())
)

alpha189['abs_deviation'] = (
    alpha189['close_filled'] - alpha189['close_mean6']
).abs()

alpha189['alpha189'] = (
    alpha189.groupby('ts_code', sort=False)['abs_deviation']
    .transform(lambda s: s.rolling(6, min_periods=6).mean())
)

alpha189.loc[alpha189['suspend'] == True, 'alpha189'] = np.nan

In [ ]:
alpha189 = alpha189[['trade_date', 'ts_code', 'alpha189']]

In [ ]:
alpha189.head()

In [ ]:
alpha189['alpha189'] = (
    alpha189.groupby('trade_date')['alpha189'].transform(wz)
)

alpha_exp_lis.append(alpha189)

## 成交活跃度与量价波动

### Alpha80

**衡量内容（原始值）：** 衡量当天成交量相对 5 个交易日前的增减幅度（百分数）。原始值为正表示放量，为负表示缩量，数值越高表示相对成交活跃度上升越明显；不直接表示买卖方向，也不等于价格涨跌幅。

$$
\mathrm{Alpha80}_{i,t}=100\left(\frac{V_{i,t}}{V_{i,t-5}}-1\right)
$$

$V$ 为成交量 `vol`。缺失成交量已在 `process.ipynb` 中填为 0，按股票滞后 5 个交易日。滞后成交量为 0 时分母无定义，因子设为 `NaN`；当天停牌时按 `suspend` 将因子设为 `NaN`。加入列表前按 `trade_date` 执行截面 `wz`。

**最少记录数：** 至少需要 6 个交易日记录；当天非停牌，且当天及第 5 个交易日前成交量有效、分母不为 0。记录数按市场交易日计，包含停牌日；上述条件指原始因子可计算，截面 `wz` 还需当日有效样本的标准差非零。

In [ ]:
alpha80 = panel[['trade_date', 'ts_code', 'vol', 'suspend']].copy()

In [ ]:
alpha80['vol_lag5'] = (
    alpha80.groupby('ts_code', sort=False)['vol'].shift(5)
)

alpha80['alpha80'] = (
    alpha80['vol'] / alpha80['vol_lag5'].replace(0, np.nan) - 1
) * 100

alpha80.loc[alpha80['suspend'] == True, 'alpha80'] = np.nan

In [ ]:
alpha80 = alpha80[['trade_date', 'ts_code', 'alpha80']]

In [ ]:
alpha80.head()

In [ ]:
alpha80['alpha80'] = (
    alpha80.groupby('trade_date')['alpha80'].transform(wz)
)

alpha_exp_lis.append(alpha80)

### Alpha95

**衡量内容（原始值）：** 衡量最近 20 个交易日成交额的绝对波动程度，单位与成交额相同（千元）。原始值越高，表示成交额在窗口内变化越大；由于没有除以平均成交额，也容易受到股票成交规模影响。停牌日的零成交额会影响窗口内的波动。

$$
\widetilde A_{i,t}=\begin{cases}0,&\text{停牌日}\\A_{i,t},&\text{其他交易日}\end{cases}
$$

$$
\overline A^{(20)}_{i,t}=\frac{1}{20}\sum_{k=0}^{19}\widetilde A_{i,t-k},\qquad
\mathrm{Alpha95}_{i,t}=\sqrt{\frac{1}{20}\sum_{k=0}^{19}\left(\widetilde A_{i,t-k}-\overline A^{(20)}_{i,t}\right)^2}
$$

$A$ 为成交额 `amount`（千元）。缺失成交额已在 `process.ipynb` 中统一填为 0，本笔记本不再填充。按股票计算包含当天的 20 日滚动标准差（`ddof=0`），使用 `min_periods=20`；停牌日的 0 参与历史窗口统计。计算后按 `suspend == True` 将当天因子设为 `NaN`，在加入列表前按 `trade_date` 执行截面 `wz`。

**最少记录数：** 至少需要 20 个交易日记录；预处理后窗口内须有 20 个有效成交额（停牌日为 0），且当天非停牌。记录数按市场交易日计，包含停牌日；上述条件指原始因子可计算，截面 `wz` 还需当日有效样本的标准差非零。

In [ ]:
alpha95 = panel[['trade_date', 'ts_code', 'amount', 'suspend']].copy()

In [ ]:
alpha95['alpha95'] = (
    alpha95.groupby('ts_code', sort=False)['amount']
    .transform(
        lambda s: s.rolling(20, min_periods=20).std(ddof=0)
    )
)

alpha95.loc[alpha95['suspend'] == True, 'alpha95'] = np.nan

In [ ]:
alpha95 = alpha95[['trade_date', 'ts_code', 'alpha95']]

In [ ]:
alpha95.head()

In [ ]:
alpha95['alpha95'] = (
    alpha95.groupby('trade_date')['alpha95'].transform(wz)
)

alpha_exp_lis.append(alpha95)

### Alpha132

**衡量内容（原始值）：** 衡量最近 20 个交易日的平均成交额，单位为千元。原始值越高，表示平均交易金额越大，用于描述成交规模与交易活跃程度；它不是换手率，也未按市值调整。停牌日以 0 参与平均，会降低窗口均值。

$$
\widetilde A_{i,t}=\begin{cases}0,&\mathrm{suspend}_{i,t}=\mathrm{True},\\A_{i,t},&\mathrm{otherwise}.\end{cases}
$$

$$
\mathrm{Alpha132}^{\mathrm{raw}}_{i,t}=\frac{1}{20}\sum_{k=0}^{19}\widetilde A_{i,t-k}
$$

$A$ 为成交额 `amount`（千元）。缺失成交额已在 `process.ipynb` 中统一填为 0，本笔记本不再填充。按股票计算包含当天的 20 日滚动均值，使用 `min_periods=20`；停牌日的 0 参与历史窗口统计。计算后按 `suspend == True` 将当天因子设为 `NaN`，在加入列表前按 `trade_date` 执行截面 `wz`。

**最少记录数：** 至少需要 20 个交易日记录；预处理后窗口内须有 20 个有效成交额（停牌日为 0），且当天非停牌。记录数按市场交易日计，包含停牌日；上述条件指原始因子可计算，截面 `wz` 还需当日有效样本的标准差非零。

In [ ]:
alpha132 = panel[['trade_date', 'ts_code', 'amount', 'suspend']].copy()

In [ ]:
alpha132['alpha132'] = (
    alpha132.groupby('ts_code', sort=False)['amount']
    .transform(
        lambda s: s.rolling(20, min_periods=20).mean()
    )
)

alpha132.loc[alpha132['suspend'] == True, 'alpha132'] = np.nan

In [ ]:
alpha132 = alpha132[['trade_date', 'ts_code', 'alpha132']]

In [ ]:
alpha132.head()

In [ ]:
alpha132['alpha132'] = (
    alpha132.groupby('trade_date')['alpha132'].transform(wz)
)

alpha_exp_lis.append(alpha132)

### Alpha76

**衡量内容（原始值）：** 衡量最近 20 个市场交易日中，单位成交量对应的绝对收益率的不稳定程度。它是该序列的变异系数：值越高表示相对波动越大，值越低表示相对稳定。不区分涨跌方向，也不直接衡量单位成交量价格变动的平均水平；它使用成交量，而非成交额。

$$
X_{i,t}=\begin{cases}
0,&\mathrm{suspend}_{i,t}=\mathrm{True},\\
\dfrac{|R_{i,t}|}{V_{i,t}},&\text{其他情况}.
\end{cases}
$$

$$
\mu_{i,t}^{(20)}=\frac{1}{20}\sum_{k=0}^{19}X_{i,t-k},\qquad
\mathrm{Alpha76}^{\mathrm{raw}}_{i,t}
=\frac{\sqrt{\frac{1}{20}\sum_{k=0}^{19}(X_{i,t-k}-\mu_{i,t}^{(20)})^2}}
{\mu_{i,t}^{(20)}}
$$

$R$ 为 `panel.ret`，$V$ 为成交量 `vol`。收益率使用上游按股票前向填充收盘价后计算的结果；成交量的缺失值已在 `process.ipynb` 中填为 0，本笔记本不再填充。停牌日的 $0/0$ 原本无定义，本项目明确约定仅在滚动统计时将当天 $X$ 设为 0，因此停牌日会影响均值与标准差。非停牌日缺失收益率保持缺失，不填为 0。复牌日收益率相对停牌前最后有效收盘价计算，可能跨越多个交易日，而成交量仅对应复牌当天。

按股票计算包含当天的完整 20 日窗口（`min_periods=20`），标准差采用 `ddof=0`。成交量分母为 0 时先保留缺失，再按停牌标记设置上述零贡献；20 日均值为 0 时，因子设为 `NaN`。计算后按 `suspend == True` 将当天因子设为 `NaN`，加入列表前按 `trade_date` 执行截面 `wz`。

**最少记录数：** 通常至少需要 21 个交易日价格记录，形成 20 个有效收益率；窗口内停牌日按零贡献计入，其余日期须有有效收益率和正成交量，且当天非停牌、窗口均值大于 0。截面 `wz` 还需当日有效样本的标准差非零。

In [ ]:
alpha76 = panel[['trade_date', 'ts_code', 'ret', 'vol', 'suspend']].copy()

In [ ]:
alpha76['abs_ret_per_vol'] = (
    alpha76['ret'].abs() / alpha76['vol'].replace(0, np.nan)
)
alpha76.loc[alpha76['suspend'] == True, 'abs_ret_per_vol'] = 0.0

alpha76['ratio_mean20'] = (
    alpha76.groupby('ts_code', sort=False)['abs_ret_per_vol']
    .transform(lambda s: s.rolling(20, min_periods=20).mean())
)

alpha76['ratio_std20'] = (
    alpha76.groupby('ts_code', sort=False)['abs_ret_per_vol']
    .transform(lambda s: s.rolling(20, min_periods=20).std(ddof=0))
)

alpha76['alpha76'] = (
    alpha76['ratio_std20'] / alpha76['ratio_mean20'].replace(0, np.nan)
)

alpha76.loc[alpha76['suspend'] == True, 'alpha76'] = np.nan

In [ ]:
alpha76 = alpha76[['trade_date', 'ts_code', 'alpha76']]

In [ ]:
alpha76.head()

In [ ]:
alpha76['alpha76'] = (
    alpha76.groupby('trade_date')['alpha76'].transform(wz)
)

alpha_exp_lis.append(alpha76)

## 涨跌方向与成交量分布

### Alpha40

**衡量内容（原始值）：** 衡量最近 26 个市场交易日中，上涨日累计成交量相对于非上涨日（下跌或平盘）累计成交量的大小。原始值为 100 表示两者相等，为 200 表示上涨日成交量为非上涨日的两倍。值越高，成交量越集中于上涨日；不衡量涨跌幅，也不等于资金净流入。与 Alpha53 的上涨天数占比相比，本因子按成交量衡量上涨日与非上涨日的相对活跃程度。

$$
\widetilde C_{i,t}=\operatorname{FFill}_i(C_{i,t}),\qquad
U_{i,t}=\mathbf{1}(\widetilde C_{i,t}>\widetilde C_{i,t-1})
$$

$$
\mathrm{Alpha40}^{\mathrm{raw}}_{i,t}
=100\frac{\sum_{k=0}^{25}V_{i,t-k}U_{i,t-k}}
{\sum_{k=0}^{25}V_{i,t-k}(1-U_{i,t-k})}
$$

$C$ 为后复权收盘价 `close_hfq`，$V$ 为成交量 `vol`。按股票前向填充收盘价后判断涨跌，复牌日与停牌前最后有效收盘价比较；尚无有效参考价格时，上涨标记保留 `NaN`，不归入非上涨日。成交量已在 `process.ipynb` 中将缺失值填为 0，本笔记本不再填充。停牌日成交量为 0，不增加分子或分母，但仍占用市场交易日窗口。

使用包含当天的完整 26 日窗口（`min_periods=26`）。分母为 0 时结果设为 `NaN`；按 `suspend == True` 将当天停牌因子设为 `NaN`。加入列表前按 `trade_date` 执行截面 `wz`。

**最少记录数：** 至少需要 28 个交易日记录，形成 26 次有效价格比较；窗口内分类与成交量须有效，非上涨日累计成交量大于 0，且当天非停牌。截面 `wz` 还需当日有效样本的标准差非零。

In [ ]:
alpha40 = panel[['trade_date', 'ts_code', 'close_hfq', 'vol', 'suspend']].copy()

In [ ]:
alpha40['close_filled'] = (
    alpha40.groupby('ts_code', sort=False)['close_hfq'].ffill()
)

alpha40['close_lag1'] = (
    alpha40.groupby('ts_code', sort=False)['close_filled'].shift(1)
)

alpha40['up'] = (
    alpha40['close_filled'] > alpha40['close_lag1']
).astype(float)

alpha40.loc[
    alpha40['close_filled'].isna() | alpha40['close_lag1'].isna(),
    'up'
] = np.nan

alpha40['up_vol'] = alpha40['vol'] * alpha40['up']
alpha40['non_up_vol'] = alpha40['vol'] * (1 - alpha40['up'])

alpha40['up_vol_sum26'] = (
    alpha40.groupby('ts_code', sort=False)['up_vol']
    .transform(lambda s: s.rolling(26, min_periods=26).sum())
)

alpha40['non_up_vol_sum26'] = (
    alpha40.groupby('ts_code', sort=False)['non_up_vol']
    .transform(lambda s: s.rolling(26, min_periods=26).sum())
)

alpha40['alpha40'] = (
    alpha40['up_vol_sum26']
    / alpha40['non_up_vol_sum26'].replace(0, np.nan)
) * 100

alpha40.loc[alpha40['suspend'] == True, 'alpha40'] = np.nan

In [ ]:
alpha40 = alpha40[['trade_date', 'ts_code', 'alpha40']]

In [ ]:
alpha40.head()

In [ ]:
alpha40['alpha40'] = (
    alpha40.groupby('trade_date')['alpha40'].transform(wz)
)

alpha_exp_lis.append(alpha40)

### Alpha43

**衡量内容（原始值）：** 衡量最近 6 个市场交易日中，上涨日累计成交量减去下跌日累计成交量的差额，单位为手。原始值为正表示上涨日累计成交量更大，为负表示下跌日累计成交量更大；平盘日贡献为 0。该指标反映短期成交量在涨跌日之间的分布，不等于资金净流入，且受股票成交规模影响。

$$
\widetilde C_{i,t}=\operatorname{FFill}_i(C_{i,t}),\qquad
S_{i,t}=\operatorname{Sign}(\widetilde C_{i,t}-\widetilde C_{i,t-1})
$$

$$
\mathrm{Alpha43}^{\mathrm{raw}}_{i,t}
=\sum_{k=0}^{5}V_{i,t-k}S_{i,t-k}
$$

$C$ 为后复权收盘价 `close_hfq`，$V$ 为成交量 `vol`。按股票前向填充收盘价后判断涨跌：上涨取 1，下跌取 -1，平盘取 0；复牌日与停牌前最后有效收盘价比较。首次缺少参考价格时，方向标记保持 `NaN`。成交量的缺失值已在 `process.ipynb` 中填为 0，本笔记本不再填充。

使用包含当天的完整 6 日窗口（`min_periods=6`），不跳过停牌日；停牌日成交量为 0，对历史窗口求和贡献为 0。计算后按 `suspend == True` 将当天因子设为 `NaN`，加入列表前按 `trade_date` 执行截面 `wz`。

**最少记录数：** 至少需要 8 个交易日记录，形成 6 次有效价格比较；窗口内方向标记与成交量须有效，且当天非停牌。截面 `wz` 还需当日有效样本的标准差非零。

In [ ]:
alpha43 = panel[['trade_date', 'ts_code', 'close_hfq', 'vol', 'suspend']].copy()

In [ ]:
alpha43['close_filled'] = (
    alpha43.groupby('ts_code', sort=False)['close_hfq'].ffill()
)

alpha43['price_change'] = (
    alpha43.groupby('ts_code', sort=False)['close_filled'].diff()
)

alpha43['signed_vol'] = alpha43['vol'] * np.sign(alpha43['price_change'])

alpha43['alpha43'] = (
    alpha43.groupby('ts_code', sort=False)['signed_vol']
    .transform(lambda s: s.rolling(6, min_periods=6).sum())
)

alpha43.loc[alpha43['suspend'] == True, 'alpha43'] = np.nan

In [ ]:
alpha43 = alpha43[['trade_date', 'ts_code', 'alpha43']]

In [ ]:
alpha43.head()

In [ ]:
alpha43['alpha43'] = (
    alpha43.groupby('trade_date')['alpha43'].transform(wz)
)

alpha_exp_lis.append(alpha43)

### Alpha84

**衡量内容（原始值）：** 衡量最近 20 个市场交易日中，上涨日累计成交量减去下跌日累计成交量的差额，单位为手。原始值为正表示上涨日累计成交量更大，为负表示下跌日累计成交量更大；平盘日贡献为 0。与 Alpha43 的定义相同，但窗口从 6 日延长至 20 日，描述更长时间内的涨跌日成交量分布。该指标未按总成交量归一化，受股票成交规模影响，不等于资金净流入。

$$
\widetilde C_{i,t}=\operatorname{FFill}_i(C_{i,t}),\qquad
S_{i,t}=\operatorname{Sign}(\widetilde C_{i,t}-\widetilde C_{i,t-1})
$$

$$
\mathrm{Alpha84}^{\mathrm{raw}}_{i,t}
=\sum_{k=0}^{19}V_{i,t-k}S_{i,t-k}
$$

$C$ 为后复权收盘价 `close_hfq`，$V$ 为成交量 `vol`。按股票前向填充收盘价后判断涨跌：上涨取 1，下跌取 -1，平盘取 0；复牌日与停牌前最后有效收盘价比较。首次缺少参考价格时，方向标记保持 `NaN`。成交量的缺失值已在 `process.ipynb` 中填为 0，本笔记本不再填充。

使用包含当天的完整 20 日窗口（`min_periods=20`），不跳过停牌日；停牌日成交量为 0，对历史窗口求和贡献为 0。计算后按 `suspend == True` 将当天因子设为 `NaN`，加入列表前按 `trade_date` 执行截面 `wz`。

**最少记录数：** 至少需要 21 个交易日记录，形成 20 次有效价格比较；窗口内方向标记与成交量须有效，且当天非停牌。截面 `wz` 还需当日有效样本的标准差非零。

In [ ]:
alpha84 = panel[['trade_date', 'ts_code', 'close_hfq', 'vol', 'suspend']].copy()

In [ ]:
alpha84['close_filled'] = (
    alpha84.groupby('ts_code', sort=False)['close_hfq'].ffill()
)

alpha84['price_change'] = (
    alpha84.groupby('ts_code', sort=False)['close_filled'].diff()
)

alpha84['signed_vol'] = alpha84['vol'] * np.sign(alpha84['price_change'])

alpha84['alpha84'] = (
    alpha84.groupby('ts_code', sort=False)['signed_vol']
    .transform(lambda s: s.rolling(20, min_periods=20).sum())
)

alpha84.loc[alpha84['suspend'] == True, 'alpha84'] = np.nan

In [ ]:
alpha84 = alpha84[['trade_date', 'ts_code', 'alpha84']]

In [ ]:
alpha84.head()

In [ ]:
alpha84['alpha84'] = (
    alpha84.groupby('trade_date')['alpha84'].transform(wz)
)

alpha_exp_lis.append(alpha84)

### Alpha60

**衡量内容（原始值）：** 衡量最近 20 个市场交易日中，收盘价相对日内高低价区间的位置，并以成交量加权累计。收盘靠近最高价时贡献为正，靠近最低价时贡献为负，位于高低价中点时贡献为 0；成交量越大，当天贡献的绝对值越大。原始值为正表示窗口内成交量加权的收盘位置整体偏强，为负表示整体偏弱。该指标未除以总成交量，受股票成交规模影响，不等于资金净流入。

$$
P_{i,t}=\frac{(C_{i,t}-L_{i,t})-(H_{i,t}-C_{i,t})}{H_{i,t}-L_{i,t}}
$$

$$
B_{i,t}=\begin{cases}
0,&\mathrm{suspend}_{i,t}=\mathrm{True},\\
V_{i,t}P_{i,t},&\text{其他情况},
\end{cases}
\qquad
\mathrm{Alpha60}^{\mathrm{raw}}_{i,t}=\sum_{k=0}^{19}B_{i,t-k}
$$

$C,H,L$ 分别为 `close_hfq`、`high_hfq`、`low_hfq`，$V$ 为成交量 `vol`。不填充价格或收盘位置指标；成交量缺失值已在 `process.ipynb` 中填为 0。滚动统计时，停牌日贡献明确设为 0，避免缺失价格乘以零仍产生 `NaN`；非停牌日若最高价等于最低价，则位置指标无定义，保留 `NaN`，不人为加小常数。

按股票计算包含当天的完整 20 日窗口（`min_periods=20`），不跳过停牌日。计算后按 `suspend == True` 将当天因子设为 `NaN`，加入列表前按 `trade_date` 执行截面 `wz`。

**最少记录数：** 至少需要 20 个交易日记录；窗口内非停牌日的高、低、收盘价与成交量须有效，且最高价不等于最低价，停牌日按零贡献计入窗口；当天须非停牌。截面 `wz` 还需当日有效样本的标准差非零。

In [ ]:
alpha60 = panel[['trade_date', 'ts_code', 'close_hfq', 'high_hfq', 'low_hfq', 'vol', 'suspend']].copy()

In [ ]:
alpha60['close_position'] = (
    (2 * alpha60['close_hfq'] - alpha60['high_hfq'] - alpha60['low_hfq'])
    / (alpha60['high_hfq'] - alpha60['low_hfq']).replace(0, np.nan)
)

alpha60['weighted_position'] = alpha60['vol'] * alpha60['close_position']
alpha60.loc[alpha60['suspend'] == True, 'weighted_position'] = 0.0

alpha60['alpha60'] = (
    alpha60.groupby('ts_code', sort=False)['weighted_position']
    .transform(lambda s: s.rolling(20, min_periods=20).sum())
)

alpha60.loc[alpha60['suspend'] == True, 'alpha60'] = np.nan

In [ ]:
alpha60 = alpha60[['trade_date', 'ts_code', 'alpha60']]

In [ ]:
alpha60.head()

In [ ]:
alpha60['alpha60'] = (
    alpha60.groupby('trade_date')['alpha60'].transform(wz)
)

alpha_exp_lis.append(alpha60)

## 量价排名与相关性

### Alpha1

**衡量内容（原始值）：** 衡量最近 6 个交易日中，成交量增速的截面排名与日内收益率的截面排名是否同步变化，再取相关系数的负值。原始值接近 -1 表示两者同向变化，接近 1 表示反向变化，接近 0 表示线性关系较弱。成交量增速不是成交量规模，日内收益率也不是收盘到收盘收益率；高值不直接表示放量上涨，预测方向须通过后续检验判断。

$$
G_{i,t}=\ln V_{i,t}-\ln V_{i,t-1},\qquad
B_{i,t}=\frac{C_{i,t}-O_{i,t}}{O_{i,t}}
$$

$$
\mathrm{Alpha1}^{\mathrm{raw}}_{i,t}
=-\operatorname{Corr}_{6}\left(
\operatorname{Rank}_t(G_{i,t}),
\operatorname{Rank}_t(B_{i,t})
\right)
$$

$V$ 为 `vol`，$C,O$ 分别为 `close_hfq`、`open_hfq`。不填充成交量或价格。零成交量先转为 `NaN` 再取对数，按股票计算一阶差分；当天停牌时，对数成交量和日内收益率均设为空，不参与相应截面排名。开盘价为 0 时日内收益率无定义，保留为空。

两个 `Rank` 均按 `trade_date` 对有效股票分别计算百分位排名，使用平均并列排名（`method='average', pct=True`）；随后按股票计算包含当天的完整 6 日配对滚动相关系数（`min_periods=6`），不是在同一天跨股票计算相关。窗口内有效配对不足 6 个或任一排名序列为常数时结果为空。当天停牌因子按 `suspend` 置为 `NaN`，加入列表前再按 `trade_date` 执行截面 `wz`。

**最少记录数：** 至少需要 8 个交易日记录，形成 6 次有效成交量变化与排名配对。复牌当天因前一日成交量为 0，无法形成有效对数变化；复牌当天算第 1 日，最早在第 8 个交易日恢复计算，前提是期间连续有效且排名序列均非常数。截面 `wz` 还需当日有效样本的标准差非零。

In [ ]:
alpha1 = panel[['trade_date', 'ts_code', 'open_hfq', 'close_hfq', 'vol', 'suspend']].copy()

In [ ]:
alpha1['log_vol'] = np.log(alpha1['vol'].replace(0, np.nan))
alpha1.loc[alpha1['suspend'] == True, 'log_vol'] = np.nan

alpha1['delta_log_vol'] = (
    alpha1.groupby('ts_code', sort=False)['log_vol'].diff()
)

alpha1['intraday_ret'] = (
    (alpha1['close_hfq'] - alpha1['open_hfq'])
    / alpha1['open_hfq'].replace(0, np.nan)
)
alpha1.loc[alpha1['suspend'] == True, 'intraday_ret'] = np.nan

alpha1['vol_rank'] = (
    alpha1.groupby('trade_date')['delta_log_vol']
    .rank(method='average', pct=True)
)

alpha1['intraday_rank'] = (
    alpha1.groupby('trade_date')['intraday_ret']
    .rank(method='average', pct=True)
)

alpha1['alpha1'] = np.nan
for _, group in alpha1.groupby('ts_code', sort=False):
    alpha1.loc[group.index, 'alpha1'] = -(
        rolling_corr(group['vol_rank'], group['intraday_rank'], 6)
    )

alpha1.loc[alpha1['suspend'] == True, 'alpha1'] = np.nan

In [ ]:
alpha1 = alpha1[['trade_date', 'ts_code', 'alpha1']]

In [ ]:
alpha1.head()

In [ ]:
alpha1['alpha1'] = (
    alpha1.groupby('trade_date')['alpha1'].transform(wz)
)

alpha_exp_lis.append(alpha1)

### Alpha5

**衡量内容（原始值）：** 衡量成交量和最高价在各自近期窗口内的相对位置是否同步变化，再取最近 3 个相关系数的最大值并反向打分。原始值接近 -1，表示最近三个相关窗口中至少一个呈现较强的量价排名同向关系；接近 1，表示这三个窗口都呈现较强反向关系。它使用最高价而非收益率，不直接表示放量上涨，也不保证后续发生价格反转。

$$
Q_{i,t}^{V}=\operatorname{TSRank}_{5}(V)_{i,t},\qquad
Q_{i,t}^{H}=\operatorname{TSRank}_{5}(H)_{i,t}
$$

$$
\rho_{i,t}=\operatorname{Corr}_{5}(Q^V,Q^H)_{i,t},\qquad
\mathrm{Alpha5}^{\mathrm{raw}}_{i,t}
=-\max_{k=0,1,2}\rho_{i,t-k}
$$

$V$ 为 `vol`，$H$ 为 `high_hfq`。`TSRank` 是同一股票最近 5 个市场交易日中当前值的百分位排名，使用平均并列排名（`method='average', pct=True`），不是跨股票排名。

成交量缺失值已在 `process.ipynb` 中填为 0，停牌日的 0 参与成交量排名；最高价不填充，含缺失最高价的 5 日窗口不输出最高价排名。两组排名再按股票计算完整 5 日配对相关系数，最后取完整 3 日窗口的最大值并取负号。三层窗口均设置完整的 `min_periods`；任一排名序列为常数时相关系数为空，外层最大值不跳过缺失凑窗口。

计算后按 `suspend == True` 将当天因子设为 `NaN`，加入列表前按 `trade_date` 执行截面 `wz`。

**最少记录数：** 至少需要 11 个交易日记录：第 5 日首次得到时序排名，第 9 日首次得到 5 日相关系数，第 11 日首次得到连续 3 个相关系数的最大值。停牌后最高价不填充，最早在复牌第 11 个交易日恢复计算，前提是期间连续有效，且各相关窗口内两个排名序列均非常数。截面 `wz` 还需当日有效样本的标准差非零。

In [ ]:
alpha5 = panel[['trade_date', 'ts_code', 'vol', 'high_hfq', 'suspend']].copy()

In [ ]:
alpha5['vol_tsrank5'] = (
    alpha5.groupby('ts_code', sort=False)['vol']
    .transform(
        lambda s: s.rolling(5, min_periods=5).rank(method='average', pct=True)
    )
)

alpha5['high_tsrank5'] = (
    alpha5.groupby('ts_code', sort=False)['high_hfq']
    .transform(
        lambda s: s.rolling(5, min_periods=5).rank(method='average', pct=True)
    )
)

alpha5['rank_corr5'] = np.nan
for _, group in alpha5.groupby('ts_code', sort=False):
    alpha5.loc[group.index, 'rank_corr5'] = (
        rolling_corr(group['vol_tsrank5'], group['high_tsrank5'], 5)
    )

alpha5['alpha5'] = -(
    alpha5.groupby('ts_code', sort=False)['rank_corr5']
    .transform(lambda s: s.rolling(3, min_periods=3).max())
)

alpha5.loc[alpha5['suspend'] == True, 'alpha5'] = np.nan

In [ ]:
alpha5 = alpha5[['trade_date', 'ts_code', 'alpha5']]

In [ ]:
alpha5.head()

In [ ]:
alpha5['alpha5'] = (
    alpha5.groupby('trade_date')['alpha5'].transform(wz)
)

alpha_exp_lis.append(alpha5)

### Alpha12

**衡量内容（原始值）：** 同时衡量当天开盘价相对近期平均 VWAP 的偏离排名，以及收盘价偏离当天 VWAP 的绝对距离排名，再将两者相乘取负。开盘价相对近期平均 VWAP 越高、收盘价与当天 VWAP 的距离越大，原始值越低。第二项不区分收盘价高于还是低于 VWAP；第一项按有符号偏离排名。它使用价格差的截面排名，未先按价格水平归一化，不直接表示日内收益或确定的反转预测。

$$
\widetilde W_{i,t}=\operatorname{FFill}_i(W_{i,t}),\qquad
\overline W_{i,t}^{(10)}=\frac{1}{10}\sum_{k=0}^{9}\widetilde W_{i,t-k}
$$

$$
\mathrm{Alpha12}^{\mathrm{raw}}_{i,t}
=-\operatorname{Rank}_t(O_{i,t}-\overline W_{i,t}^{(10)})
\operatorname{Rank}_t(|C_{i,t}-W_{i,t}|)
$$

$O,C,W$ 分别为 `open_hfq`、`close_hfq`、`vwap_hfq`。仅用于 10 日滚动均值的 VWAP 按股票前向填充，使用 `min_periods=10`；当天开盘价、收盘价和计算第二项的当天 VWAP 保持原值。首次有效 VWAP 之前不填充。停牌日两项排名输入置空，不参与当日排名。

两个 Rank 分别按 `trade_date` 对有效值计算百分位排名，使用平均并列排名（`method='average', pct=True`）。当天停牌因子置为 `NaN`，加入列表前按交易日执行截面 `wz`。

**最少记录数：** 至少需要 10 个交易日记录；前向填充后窗口内须有 10 个有效 VWAP，当天 O、C、W 有效且非停牌。截面 `wz` 还需当日有效样本的标准差非零。

In [ ]:
alpha12 = panel[['trade_date', 'ts_code', 'open_hfq', 'close_hfq', 'vwap_hfq', 'suspend']].copy()

In [ ]:
alpha12['vwap_mean10'] = (
    alpha12.groupby('ts_code', sort=False)['vwap_hfq']
    .transform(lambda s: s.ffill().rolling(10, min_periods=10).mean())
)

alpha12['open_deviation'] = alpha12['open_hfq'] - alpha12['vwap_mean10']
alpha12['close_vwap_distance'] = (alpha12['close_hfq'] - alpha12['vwap_hfq']).abs()
alpha12.loc[
    alpha12['suspend'] == True, ['open_deviation', 'close_vwap_distance']
] = np.nan

alpha12['open_rank'] = (
    alpha12.groupby('trade_date')['open_deviation'].rank(method='average', pct=True)
)
alpha12['distance_rank'] = (
    alpha12.groupby('trade_date')['close_vwap_distance'].rank(method='average', pct=True)
)

alpha12['alpha12'] = -alpha12['open_rank'] * alpha12['distance_rank']
alpha12.loc[alpha12['suspend'] == True, 'alpha12'] = np.nan

In [ ]:
alpha12 = alpha12[['trade_date', 'ts_code', 'alpha12']]

In [ ]:
alpha12.head()

In [ ]:
alpha12['alpha12'] = (
    alpha12.groupby('trade_date')['alpha12'].transform(wz)
)

alpha_exp_lis.append(alpha12)

### Alpha42

**衡量内容（原始值）：** 将最高价近 10 日标准差的截面排名，与最高价和成交量近 10 日的相关系数相乘，再取负号。最高价绝对波动排名较高且量价正相关时，原始值更负；量价负相关时为正，波动排名放大其幅度。它衡量的是最高价波动与量价联动的组合，不直接衡量收益率；最高价标准差未按价格归一化，受价格水平及复权尺度影响。

$$
\mathrm{Alpha42}^{\mathrm{raw}}_{i,t}
=-\operatorname{Rank}_t\left(\operatorname{Std}_{10}(H)_{i,t}\right)
\operatorname{Corr}_{10}(H,V)_{i,t}
$$

$H$ 为 `high_hfq`，$V$ 为 `vol`。最高价不填充；成交量沿用上游填零结果。按股票使用完整 10 日窗口（`min_periods=10`）计算最高价总体标准差（`ddof=0`）和最高价、成交量的配对相关。窗口中缺少最高价会导致相关系数缺失；任一相关序列为常数时结果为空。

最高价标准差先将当天停牌记录置空，再按 `trade_date` 计算平均并列百分位排名（`method='average', pct=True`）。最终停牌因子置为 `NaN`，加入列表前按交易日执行截面 `wz`。

**最少记录数：** 至少需要 10 个交易日记录；窗口内高价、成交量须有效，两组相关序列均非常数，且当天非停牌。因最高价不填充，最早在连续有效的复牌第 10 日恢复计算。截面 `wz` 还需当日有效样本的标准差非零。

In [ ]:
alpha42 = panel[['trade_date', 'ts_code', 'high_hfq', 'vol', 'suspend']].copy()

In [ ]:
alpha42['high_std10'] = (
    alpha42.groupby('ts_code', sort=False)['high_hfq']
    .transform(lambda s: s.rolling(10, min_periods=10).std(ddof=0))
)
alpha42.loc[alpha42['suspend'] == True, 'high_std10'] = np.nan

alpha42['std_rank'] = (
    alpha42.groupby('trade_date')['high_std10'].rank(method='average', pct=True)
)

alpha42['high_vol_corr10'] = np.nan
for _, group in alpha42.groupby('ts_code', sort=False):
    alpha42.loc[group.index, 'high_vol_corr10'] = (
        rolling_corr(group['high_hfq'], group['vol'], 10)
    )

alpha42['alpha42'] = -alpha42['std_rank'] * alpha42['high_vol_corr10']
alpha42.loc[alpha42['suspend'] == True, 'alpha42'] = np.nan

In [ ]:
alpha42 = alpha42[['trade_date', 'ts_code', 'alpha42']]

In [ ]:
alpha42.head()

In [ ]:
alpha42['alpha42'] = (
    alpha42.groupby('trade_date')['alpha42'].transform(wz)
)

alpha_exp_lis.append(alpha42)

### Alpha105

**衡量内容（原始值）：** 衡量最近 10 日内，开盘价截面排名与成交量截面排名的相关性，并取负号。原始值接近 -1 表示两组排名同向变化，接近 1 表示反向变化，接近 0 表示线性关系较弱。排名反映各自在当天股票截面的位置，不是收益率排名或成交量增速排名；本因子也不使用收盘价。

$$
\mathrm{Alpha105}^{\mathrm{raw}}_{i,t}
=-\operatorname{Corr}_{10}\left(
\operatorname{Rank}_t(O_{i,t}),\operatorname{Rank}_t(V_{i,t})
\right)
$$

$O$ 为 `open_hfq`，$V$ 为 `vol`。价格不填充，成交量沿用上游填零结果，但停牌日的开盘价、成交量排名输入均置为 `NaN`，不参与当天截面排名；这仅修改因子副本，不修改 panel。两个 Rank 分别按 `trade_date` 对有效值做平均并列百分位排名（`method='average', pct=True`）。

按股票计算包含当天的完整 10 日配对滚动相关（`min_periods=10`），不是同一天跨股票计算相关；缺少有效配对或任一排名序列为常数时结果为空。当天停牌因子置空，加入列表前按交易日执行截面 `wz`。

**最少记录数：** 至少需要 10 个交易日记录，且连续 10 日排名配对有效、两组排名序列均非常数，当天非停牌。最早在连续有效的复牌第 10 日恢复计算。截面 `wz` 还需当日有效样本的标准差非零。

In [ ]:
alpha105 = panel[['trade_date', 'ts_code', 'open_hfq', 'vol', 'suspend']].copy()

In [ ]:
alpha105.loc[alpha105['suspend'] == True, ['open_hfq', 'vol']] = np.nan

alpha105['open_rank'] = (
    alpha105.groupby('trade_date')['open_hfq'].rank(method='average', pct=True)
)
alpha105['vol_rank'] = (
    alpha105.groupby('trade_date')['vol'].rank(method='average', pct=True)
)

alpha105['alpha105'] = np.nan
for _, group in alpha105.groupby('ts_code', sort=False):
    alpha105.loc[group.index, 'alpha105'] = -(
        rolling_corr(group['open_rank'], group['vol_rank'], 10)
    )

alpha105.loc[alpha105['suspend'] == True, 'alpha105'] = np.nan

In [ ]:
alpha105 = alpha105[['trade_date', 'ts_code', 'alpha105']]

In [ ]:
alpha105.head()

In [ ]:
alpha105['alpha105'] = (
    alpha105.groupby('trade_date')['alpha105'].transform(wz)
)

alpha_exp_lis.append(alpha105)

### Alpha139

**衡量内容（原始值）：** 衡量最近 10 个交易日开盘价水平与成交量的线性相关性，并取负号。原始值接近 -1 表示开盘价较高的日子通常成交量也较大；接近 1 表示开盘价较高的日子成交量反而较小。与 Alpha105 不同，它直接使用单只股票的开盘价和成交量，不先做每日截面排名。它不直接衡量开盘收益率或资金净流入。

$$
\mathrm{Alpha139}^{\mathrm{raw}}_{i,t}
=-\operatorname{Corr}_{10}(O,V)_{i,t}
$$

$O$ 为 `open_hfq`，$V$ 为 `vol`。开盘价不填充，成交量使用上游填零结果。按股票计算包含当天的完整 10 日配对相关（`min_periods=10`），停牌缺失开盘价使窗口内配对不完整，不跳过停牌日凑足 10 条记录。任一序列为常数时相关系数为空。当天停牌因子按 `suspend` 置空，加入列表前按交易日执行截面 `wz`。

**最少记录数：** 至少需要 10 个交易日记录；连续 10 日开盘价、成交量配对有效，两组序列均非常数，且当天非停牌。最早在连续有效的复牌第 10 日恢复计算。截面 `wz` 还需当日有效样本的标准差非零。

In [ ]:
alpha139 = panel[['trade_date', 'ts_code', 'open_hfq', 'vol', 'suspend']].copy()

In [ ]:
alpha139['alpha139'] = np.nan
for _, group in alpha139.groupby('ts_code', sort=False):
    alpha139.loc[group.index, 'alpha139'] = -(
        rolling_corr(group['open_hfq'], group['vol'], 10)
    )

alpha139.loc[alpha139['suspend'] == True, 'alpha139'] = np.nan

In [ ]:
alpha139 = alpha139[['trade_date', 'ts_code', 'alpha139']]

In [ ]:
alpha139.head()

In [ ]:
alpha139['alpha139'] = (
    alpha139.groupby('trade_date')['alpha139'].transform(wz)
)

alpha_exp_lis.append(alpha139)

## 指数方向条件

### Alpha75

**衡量内容（原始值）：** 衡量最近 50 个市场交易日中，在指数收盘低于开盘的日子里，个股收盘高于开盘的频率。原始值介于 0 与 1 之间，越高表示指数日内走弱时个股日内收涨的比例越大，描述相对抗跌表现。这里比较开盘到收盘的方向，不使用指数或个股的收盘到收盘收益率。

$$
\mathrm{Alpha75}^{\mathrm{raw}}_{i,t}
=\frac{\sum_{k=0}^{49}\mathbf{1}(C_{i,t-k}>O_{i,t-k}\;\land\;I_{C,t-k}<I_{O,t-k})}
{\sum_{k=0}^{49}\mathbf{1}(I_{C,t-k}<I_{O,t-k})}
$$

$C,O$ 为 `close_hfq`、`open_hfq`，$I_C,I_O$ 为已读入 `index` 的 `close`、`open`，通过 `trade_date` 对齐，不按行号匹配。股票及指数价格均不填充。

采用完整 50 日窗口（`min_periods=50`）。停牌日分子贡献为 0，但当天若指数收盘低于开盘，仍计入分母；因此停牌会降低相应窗口的条件上涨频率，不从市场日历中删除停牌日。指数平盘不计入分母；分母为 0 时结果为空。非停牌日股票开、收盘价缺失时，分子输入保留缺失，不直接视为未满足条件。

当天因子按 `suspend == True` 置为 `NaN`，加入列表前按 `trade_date` 执行截面 `wz`。

**最少记录数：** 至少需要 50 个交易日记录，窗口内指数方向有效、分子计数输入有效，至少有 1 个指数日内下跌日，且当天非停牌。截面 `wz` 还需当日有效样本的标准差非零。

In [ ]:
alpha75 = panel[['trade_date', 'ts_code', 'open_hfq', 'close_hfq', 'suspend']].copy()
alpha75 = alpha75.merge(
    index[['trade_date', 'open', 'close']].rename(
        columns={'open': 'index_open', 'close': 'index_close'}
    ),
    on='trade_date', how='left'
)

In [ ]:
alpha75['index_down'] = (alpha75['index_close'] < alpha75['index_open']).astype(float)
alpha75['stock_up_index_down'] = (
    (alpha75['close_hfq'] > alpha75['open_hfq']) & (alpha75['index_down'] == 1)
).astype(float)

alpha75.loc[
    (alpha75['suspend'] == False)
    & (alpha75['open_hfq'].isna() | alpha75['close_hfq'].isna()),
    'stock_up_index_down'
] = np.nan
alpha75.loc[alpha75['suspend'] == True, 'stock_up_index_down'] = 0.0

alpha75['joint_count50'] = (
    alpha75.groupby('ts_code', sort=False)['stock_up_index_down']
    .transform(lambda s: s.rolling(50, min_periods=50).sum())
)
alpha75['index_down_count50'] = (
    alpha75.groupby('ts_code', sort=False)['index_down']
    .transform(lambda s: s.rolling(50, min_periods=50).sum())
)

alpha75['alpha75'] = (
    alpha75['joint_count50'] / alpha75['index_down_count50'].replace(0, np.nan)
)
alpha75.loc[alpha75['suspend'] == True, 'alpha75'] = np.nan

In [ ]:
alpha75 = alpha75[['trade_date', 'ts_code', 'alpha75']]

In [ ]:
alpha75.head()

In [ ]:
alpha75['alpha75'] = (
    alpha75.groupby('trade_date')['alpha75'].transform(wz)
)

alpha_exp_lis.append(alpha75)

### Alpha182

**衡量内容（原始值）：** 衡量最近 20 个市场交易日中，个股与指数日内 K 线方向一致的频率。两者同时收盘高于开盘，或同时收盘低于开盘时计为一致。原始值介于 0 与 1 之间，越高表示日内方向一致的天数越多；不区分共同上涨或共同下跌，不等于收益率相关系数。

$$
J_{i,t}=\mathbf{1}\left[
(C_{i,t}>O_{i,t}\land I_{C,t}>I_{O,t})
\lor(C_{i,t}<O_{i,t}\land I_{C,t}<I_{O,t})
\right]
$$

$$
\mathrm{Alpha182}^{\mathrm{raw}}_{i,t}=\frac{1}{20}\sum_{k=0}^{19}J_{i,t-k}
$$

$C,O$ 为 `close_hfq`、`open_hfq`，$I_C,I_O$ 为已读入 `index` 的 `close`、`open`，按 `trade_date` 对齐。比较当天开盘与收盘，不使用收盘到收盘收益率。股票及指数价格不填充；任一方平盘均不计为方向一致。

使用包含当天的完整 20 日窗口（`min_periods=20`），分母固定为 20。停牌日方向一致标记为 0，仍占用窗口，因此可能降低复牌后的频率；非停牌日股票开、收盘价缺失时，标记保持缺失，不视为方向不一致。当天因子按 `suspend == True` 置为 `NaN`，加入列表前按 `trade_date` 执行截面 `wz`。

**最少记录数：** 至少需要 20 个交易日记录，窗口内指数方向与个股条件标记有效，且当天非停牌。截面 `wz` 还需当日有效样本的标准差非零。

In [ ]:
alpha182 = panel[['trade_date', 'ts_code', 'open_hfq', 'close_hfq', 'suspend']].copy()
alpha182 = alpha182.merge(
    index[['trade_date', 'open', 'close']].rename(
        columns={'open': 'index_open', 'close': 'index_close'}
    ),
    on='trade_date', how='left'
)

In [ ]:
alpha182['same_direction'] = (
    ((alpha182['close_hfq'] > alpha182['open_hfq'])
     & (alpha182['index_close'] > alpha182['index_open']))
    | ((alpha182['close_hfq'] < alpha182['open_hfq'])
       & (alpha182['index_close'] < alpha182['index_open']))
).astype(float)

alpha182.loc[
    (alpha182['suspend'] == False)
    & (alpha182['open_hfq'].isna() | alpha182['close_hfq'].isna()),
    'same_direction'
] = np.nan
alpha182.loc[alpha182['suspend'] == True, 'same_direction'] = 0.0

alpha182['alpha182'] = (
    alpha182.groupby('ts_code', sort=False)['same_direction']
    .transform(lambda s: s.rolling(20, min_periods=20).sum())
) / 20

alpha182.loc[alpha182['suspend'] == True, 'alpha182'] = np.nan

In [ ]:
alpha182 = alpha182[['trade_date', 'ts_code', 'alpha182']]

In [ ]:
alpha182.head()

In [ ]:
alpha182['alpha182'] = (
    alpha182.groupby('trade_date')['alpha182'].transform(wz)
)

alpha_exp_lis.append(alpha182)

## 长窗口代理因子

### Alpha25：200 日代理（`alpha25_proxy200`）

**衡量内容（原始值）：** 将成交活跃度调整后的 7 日价格差反向排名，再用长期收益排名放大幅度。调整后短期价差排名越高、长期收益排名越高，原始值越负；不是简单的放量上涨指标。短期输入为绝对价格差，长期输入为日收益率算术和，不是复合累计收益率。

$$
Q_{i,t}=\frac{V_{i,t}}{\operatorname{Mean}_{20}(V)_{i,t}},\qquad
D_{i,t}=\sum_{k=0}^{8}\frac{9-k}{45}Q_{i,t-k}
$$

$$
B_{i,t}=(C_{i,t}-\widetilde C_{i,t-7})[1-\operatorname{Rank}_t(D_{i,t})]
$$

$$
\mathrm{Alpha25}^{\mathrm{proxy200}}_{i,t}
=-\operatorname{Rank}_t(B_{i,t})
\left[1+\operatorname{Rank}_t\left(\sum_{k=0}^{199}R_{i,t-k}\right)\right]
$$

$C$ 为 `close_hfq`，$\widetilde C$ 为按股票前向填充后的收盘价，仅用于滞后 7 日参考价格，当天收盘价保持原值；$V$ 为 `vol`，$R$ 为 `ret`。成交量与收益率沿用上游停牌日为 0 的口径，不再填充。20 日均量为 0 时比值为空。均量使用完整 20 日窗口，线性衰减使用完整 9 日窗口，从旧到新权重为 1 至 9，总和为 45。200 日收益率和要求完整窗口，保留最初收益率缺失。

所有 Rank 均为当日有效股票的平均并列百分位排名（`method='average', pct=True`），各层排名输入先将当天停牌股票置空。最终停牌因子为空，截面 `wz` 后加入列表。原式 250 日收益率和改为 200 日，其余参数保留，此版本不应与原始 Alpha25 混用。

**最少记录数：** 相对成交量的嵌套部分需要 28 日记录；完整 200 日收益率通常需要 201 日价格记录，因此整体通常至少需要 201 个交易日记录。当天非停牌，各依赖窗口须有效；截面 `wz` 还需当日有效样本的标准差非零。

In [ ]:
alpha25_proxy200 = panel[['trade_date', 'ts_code', 'close_hfq', 'vol', 'ret', 'suspend']].copy()

In [ ]:
alpha25_proxy200['close_lag7'] = (
    alpha25_proxy200.groupby('ts_code', sort=False)['close_hfq']
    .transform(lambda s: s.ffill().shift(7))
)
alpha25_proxy200['vol_mean20'] = (
    alpha25_proxy200.groupby('ts_code', sort=False)['vol']
    .transform(lambda s: s.rolling(20, min_periods=20).mean())
)
alpha25_proxy200['relative_vol'] = (
    alpha25_proxy200['vol'] / alpha25_proxy200['vol_mean20'].replace(0, np.nan)
)
alpha25_proxy200['vol_decay9'] = (
    alpha25_proxy200.groupby('ts_code', sort=False)['relative_vol']
    .transform(lambda s: s.rolling(9, min_periods=9).apply(
        lambda values: np.dot(values, np.arange(1, 10)) / 45, raw=True
    ))
)
alpha25_proxy200.loc[alpha25_proxy200['suspend'] == True, 'vol_decay9'] = np.nan
alpha25_proxy200['vol_decay_rank'] = (
    alpha25_proxy200.groupby('trade_date')['vol_decay9'].rank(method='average', pct=True)
)
alpha25_proxy200['adjusted_delta7'] = (
    (alpha25_proxy200['close_hfq'] - alpha25_proxy200['close_lag7'])
    * (1 - alpha25_proxy200['vol_decay_rank'])
)
alpha25_proxy200['ret_sum200'] = (
    alpha25_proxy200.groupby('ts_code', sort=False)['ret']
    .transform(lambda s: s.rolling(200, min_periods=200).sum())
)
alpha25_proxy200.loc[
    alpha25_proxy200['suspend'] == True, ['adjusted_delta7', 'ret_sum200']
] = np.nan
alpha25_proxy200['delta_rank'] = (
    alpha25_proxy200.groupby('trade_date')['adjusted_delta7'].rank(method='average', pct=True)
)
alpha25_proxy200['ret_rank'] = (
    alpha25_proxy200.groupby('trade_date')['ret_sum200'].rank(method='average', pct=True)
)
alpha25_proxy200['alpha25_proxy200'] = (
    -alpha25_proxy200['delta_rank'] * (1 + alpha25_proxy200['ret_rank'])
)
alpha25_proxy200.loc[alpha25_proxy200['suspend'] == True, 'alpha25_proxy200'] = np.nan

In [ ]:
alpha25_proxy200 = alpha25_proxy200[['trade_date', 'ts_code', 'alpha25_proxy200']]

In [ ]:
alpha25_proxy200.head()

In [ ]:
alpha25_proxy200['alpha25_proxy200'] = (
    alpha25_proxy200.groupby('trade_date')['alpha25_proxy200'].transform(wz)
)

alpha_exp_lis.append(alpha25_proxy200)

### Alpha26：180 日代理（`alpha26_proxy180`）

**衡量内容（原始值）：** 将短期收盘价相对自身 7 日均价的反向偏离，与当日 VWAP 和滞后 5 日收盘价的长期相关性相加。低于均价时第一项为正，高于均价时为负；第二项随价格水平序列的相关性变化。两项分别是价格差与无量纲相关系数，数值尺度不同，不能将整体简单解释为收益率或单一强弱指标，且价格差受后复权尺度影响。

$$
\mathrm{Alpha26}^{\mathrm{proxy180}}_{i,t}
=\frac{1}{7}\sum_{k=0}^{6}\widetilde C_{i,t-k}-C_{i,t}
+\operatorname{Corr}_{180}(W_{i,t},\widetilde C_{i,t-5})
$$

$C$ 为 `close_hfq`，$W$ 为 `vwap_hfq`。按股票前向填充收盘价，仅用于完整 7 日均价与滞后 5 日参考价格；当天收盘价和 VWAP 保持原值。VWAP 不填充，相关系数要求完整 180 日配对窗口（`min_periods=180`），任一序列恒定时结果为空。因此窗口内的停牌缺失 VWAP 会使相关项缺失，直到恢复完整窗口。

原式相关窗口由 230 日改为 180 日，其余参数保留。当天停牌因子置为空，按交易日截面 `wz` 后加入列表。

**最少记录数：** 至少需要 185 个交易日记录：滞后 5 日再计算 180 日相关。收盘价参考序列和 VWAP 配对须连续有效、两组序列非常数，且当天非停牌；发生停牌后相关项最早在复牌第 180 日恢复，仍须有足够的滞后价格历史。截面 `wz` 还需当日有效样本的标准差非零。

In [ ]:
alpha26_proxy180 = panel[['trade_date', 'ts_code', 'close_hfq', 'vwap_hfq', 'suspend']].copy()

In [ ]:
alpha26_proxy180['close_mean7'] = (
    alpha26_proxy180.groupby('ts_code', sort=False)['close_hfq']
    .transform(lambda s: s.ffill().rolling(7, min_periods=7).mean())
)
alpha26_proxy180['close_lag5'] = (
    alpha26_proxy180.groupby('ts_code', sort=False)['close_hfq']
    .transform(lambda s: s.ffill().shift(5))
)
alpha26_proxy180['vwap_close_corr180'] = np.nan
for _, group in alpha26_proxy180.groupby('ts_code', sort=False):
    alpha26_proxy180.loc[group.index, 'vwap_close_corr180'] = (
        rolling_corr(group['vwap_hfq'], group['close_lag5'], 180)
    )
alpha26_proxy180['alpha26_proxy180'] = (
    alpha26_proxy180['close_mean7'] - alpha26_proxy180['close_hfq']
    + alpha26_proxy180['vwap_close_corr180']
)
alpha26_proxy180.loc[alpha26_proxy180['suspend'] == True, 'alpha26_proxy180'] = np.nan

In [ ]:
alpha26_proxy180 = alpha26_proxy180[['trade_date', 'ts_code', 'alpha26_proxy180']]

In [ ]:
alpha26_proxy180.head()

In [ ]:
alpha26_proxy180['alpha26_proxy180'] = (
    alpha26_proxy180.groupby('trade_date')['alpha26_proxy180'].transform(wz)
)

alpha_exp_lis.append(alpha26_proxy180)

### Alpha33：200 日代理（`alpha33_proxy200`）

**衡量内容（原始值）：** 将短期低点的上移或下移，与较早一段历史收益表现的截面排名、当天成交量的短期时序排名相乘。最近 5 日最低价低于 5 日前对应窗口低点时，第一项为正；高于原窗口低点时为负。收益排名与成交量排名越高，放大该价格差的幅度越大。它不是简单的长期动量，原始值也受价格尺度影响。

$$
M_{i,t}=\operatorname{TSMin}_5(L)_{i,t},\qquad
E_{i,t}=\frac{\operatorname{Sum}_{200}(R)_{i,t}-\operatorname{Sum}_{20}(R)_{i,t}}{180}
$$

$$
\mathrm{Alpha33}^{\mathrm{proxy200}}_{i,t}
=(M_{i,t-5}-M_{i,t})\operatorname{Rank}_t(E_{i,t})
\operatorname{TSRank}_5(V)_{i,t}
$$

$L$ 为 `low_hfq`，$R$ 为 `ret`，$V$ 为 `vol`。最低价不填充，5 日最低值与其滞后 5 日均按股票计算，需要两个完整窗口。收益率沿用上游停牌日为 0、复牌日相对最后有效收盘价计算的口径。200 日收益率和减去最近 20 日收益率和，再除以 180，表示较早 180 日的平均算术收益，不是复合收益率。

Rank 为按 `trade_date` 的平均并列百分位排名，排名输入先将当天停牌股票置空；TSRank 为完整 5 日窗口内当前成交量的平均并列百分位排名。成交量使用上游填零结果，停牌零量参与历史排名。最终停牌因子为空，截面 `wz` 后加入列表。

原式 240 日收益窗口改为 200 日，相应分母由 220 改为 180。该版本与原始 Alpha33 分开命名。

**最少记录数：** 低点变化部分需要 10 日记录；完整 200 日收益率通常需要 201 日价格记录，因此整体通常至少需要 201 个交易日记录。两段低价窗口和收益窗口须有效，且当天非停牌。截面 `wz` 还需当日有效样本的标准差非零。

In [ ]:
alpha33_proxy200 = panel[['trade_date', 'ts_code', 'low_hfq', 'vol', 'ret', 'suspend']].copy()

In [ ]:
alpha33_proxy200['low_min5'] = (
    alpha33_proxy200.groupby('ts_code', sort=False)['low_hfq']
    .transform(lambda s: s.rolling(5, min_periods=5).min())
)
alpha33_proxy200['low_min5_lag5'] = (
    alpha33_proxy200.groupby('ts_code', sort=False)['low_min5'].shift(5)
)
alpha33_proxy200['ret_sum200'] = (
    alpha33_proxy200.groupby('ts_code', sort=False)['ret']
    .transform(lambda s: s.rolling(200, min_periods=200).sum())
)
alpha33_proxy200['ret_sum20'] = (
    alpha33_proxy200.groupby('ts_code', sort=False)['ret']
    .transform(lambda s: s.rolling(20, min_periods=20).sum())
)
alpha33_proxy200['earlier_ret_mean180'] = (
    (alpha33_proxy200['ret_sum200'] - alpha33_proxy200['ret_sum20']) / 180
)
alpha33_proxy200.loc[alpha33_proxy200['suspend'] == True, 'earlier_ret_mean180'] = np.nan
alpha33_proxy200['ret_rank'] = (
    alpha33_proxy200.groupby('trade_date')['earlier_ret_mean180']
    .rank(method='average', pct=True)
)
alpha33_proxy200['vol_tsrank5'] = (
    alpha33_proxy200.groupby('ts_code', sort=False)['vol']
    .transform(lambda s: s.rolling(5, min_periods=5).rank(method='average', pct=True))
)
alpha33_proxy200['alpha33_proxy200'] = (
    (alpha33_proxy200['low_min5_lag5'] - alpha33_proxy200['low_min5'])
    * alpha33_proxy200['ret_rank'] * alpha33_proxy200['vol_tsrank5']
)
alpha33_proxy200.loc[alpha33_proxy200['suspend'] == True, 'alpha33_proxy200'] = np.nan

In [ ]:
alpha33_proxy200 = alpha33_proxy200[['trade_date', 'ts_code', 'alpha33_proxy200']]

In [ ]:
alpha33_proxy200.head()

In [ ]:
alpha33_proxy200['alpha33_proxy200'] = (
    alpha33_proxy200.groupby('trade_date')['alpha33_proxy200'].transform(wz)
)

alpha_exp_lis.append(alpha33_proxy200)

## 合并因子暴露与风格输入

In [ ]:
merge_keys = ['trade_date', 'ts_code']
alpha_cols = [
    col for alpha in alpha_exp_lis for col in alpha.columns
    if col not in merge_keys
]

alpha_exp = panel[merge_keys].copy()
for alpha in alpha_exp_lis:
    alpha_exp = pd.merge(
        left=alpha_exp, right=alpha, on=merge_keys,
        how='left', validate='one_to_one'
    )

In [ ]:
style_cols = ['size', 'mom', 'vol1', 'vol2', 'liq1', 'liq2', 'beta', 'value']
alpha_exp = pd.merge(
    left=alpha_exp, right=style_factor_exposure[merge_keys + style_cols],
    on=merge_keys, how='left', validate='one_to_one'
)

## Alpha 对风格因子的截面中性化

对每个交易日、每个 Alpha 分别进行带截距的等权截面 OLS 回归：

$$
\alpha_{i,t}=a_t+\boldsymbol b_t^\top\boldsymbol f_{i,t}+\varepsilon_{i,t},
\qquad
\boldsymbol f_{i,t}=(\mathrm{size},\mathrm{mom},\mathrm{vol1},\mathrm{vol2},\mathrm{liq1},\mathrm{liq2},\mathrm{beta},\mathrm{value})_{i,t}^{\top}
$$

输入 Alpha 已在此前完成截面 `wz`。使用 `LinearRegression(fit_intercept=True)`，将残差作为去除风格线性解释部分后的暴露。这是中性化，不只是减去截面均值；不引入未来收益率或收益标签，也不额外控制行业。

每个 Alpha 单独选择当日有效样本：只要求该 Alpha 与八个风格因子均为有限值，并排除当天停牌股票。不要求其他 Alpha 同时有效，不填充缺失暴露。至少需要 10 个有效样本，即多于八个风格变量加截距的 9 个参数；这是最低样本条件，不是统计稳定性的保证。风格因子预热不足时，相应股票中性化暴露保留 `NaN`。

$$
\alpha^{\mathrm{neutral}}_{i,t}
=\frac{\varepsilon_{i,t}-\overline\varepsilon_t}
{\sigma_t(\varepsilon)},\qquad \mathrm{ddof}=0
$$

残差只进行截面均值调整和总体标准差缩放，不再次调用会缩尾的 `wz`，避免非线性缩尾破坏与控制变量的样本内正交关系。残差标准差接近 0 时，该截面不输出有效暴露。原始回归样本上的残差标准化不会改变线性中性关系。

`alpha_exp` 保留合并后的标准化 Alpha 和风格输入；`alpha_exp_neutralized` 单独保存中性化结果，列为 `trade_date`、`ts_code` 与原有 33 个 Alpha 列，列名保持一致。未进入某日某个回归的记录保留 `NaN`。

In [ ]:
neutralization_data = alpha_exp.merge(
    panel[merge_keys + ['suspend']],
    on=merge_keys, how='left', validate='one_to_one'
)
alpha_exp_neutralized = alpha_exp[merge_keys].copy()
alpha_exp_neutralized[alpha_cols] = np.nan
model_neutral = LinearRegression(fit_intercept=True)

for date, group in neutralization_data.groupby('trade_date', sort=False):
    style_valid = (
        np.isfinite(group[style_cols]).all(axis=1)
        & group['suspend'].eq(False)
    )
    for col in alpha_cols:
        sample = group.loc[style_valid & np.isfinite(group[col])]
        if len(sample) <= len(style_cols) + 1:
            continue
        X = sample[style_cols]
        y = sample[col]
        model_neutral.fit(X, y)
        residual = y - model_neutral.predict(X)
        residual_std = residual.std(ddof=0)
        if not np.isfinite(residual_std) or np.isclose(residual_std, 0):
            continue
        alpha_exp_neutralized.loc[sample.index, col] = (
            (residual - residual.mean()) / residual_std
        )

## 预览中性化结果

结果包含日期、股票代码和 33 个标准化残差暴露；未满足原始计算或中性化条件的记录保留缺失。


In [ ]:
alpha_exp_neutralized.head()

## 保存 Alpha 暴露

写入 `ALPHA_FACTOR_OUTPUT`，供因子评价和收益模型使用。保存的是中性化暴露，方向反转在后续收益模型中统一执行。


In [ ]:
alpha_exp_neutralized.to_parquet(ALPHA_FACTOR_OUTPUT, index=False)